In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

from pathlib import Path

input_dir = Path("/kaggle/input")

print("=== CONNECTED DATASETS ===")

for dataset in sorted(input_dir.iterdir()):
    print(f"\nDataset: {dataset.name}")

    for item in sorted(dataset.iterdir())[:10]:
        print("  ", item.name)

print("\n=== CHECKPOINT SEARCH ===")

transfer_zips = list(
    input_dir.rglob("srgan_kaggle_transfer.zip")
)

checkpoint_files = list(
    input_dir.rglob("ckpt-27.index")
)

print("Transfer ZIP:", transfer_zips)
print("Checkpoint 27:", checkpoint_files)


In [ ]:
from pathlib import Path
import shutil
import pandas as pd
import tensorflow as tf

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------

TRANSFER_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

WORK_ROOT = Path("/kaggle/working/project")

DOGS_BASE = Path("/kaggle/input/datasets/blourdhuraju")

print("=== PATH CHECK ===")
print("TRANSFER_ROOT exists:", TRANSFER_ROOT.exists())
print("WORK_ROOT exists before copy:", WORK_ROOT.exists())
print("DOGS_BASE exists:", DOGS_BASE.exists())

# --------------------------------------------------
# 2. Find Dogs-vs-Cats dataset root
# --------------------------------------------------

dataset_info_files = list(DOGS_BASE.rglob("dataset_info.csv"))

print("\n=== DOGS DATASET SEARCH ===")
print("dataset_info.csv files found:")
for p in dataset_info_files:
    print(" ", p)

if not dataset_info_files:
    raise FileNotFoundError("Could not find dataset_info.csv for dogs-vs-cats dataset.")

DOGS_DATASET_ROOT = dataset_info_files[0].parent

print("\nSelected DOGS_DATASET_ROOT:")
print(DOGS_DATASET_ROOT)

print("\nSubfolders:")
for item in sorted(DOGS_DATASET_ROOT.iterdir()):
    print(" ", item.name)

# --------------------------------------------------
# 3. Copy transferred project to writable working dir
# --------------------------------------------------

if WORK_ROOT.exists():
    print("\nWorking directory already exists. Skipping copy.")
else:
    print("\nCopying project to /kaggle/working ...")
    shutil.copytree(TRANSFER_ROOT, WORK_ROOT)
    print("Copy completed.")

# --------------------------------------------------
# 4. Verify copied training files
# --------------------------------------------------

resume_dir = WORK_ROOT / "checkpoints" / "srgan" / "groupnorm_v2" / "resume"
archive_dir = WORK_ROOT / "checkpoints" / "srgan" / "groupnorm_v2" / "archive"
best_dir = WORK_ROOT / "checkpoints" / "srgan" / "groupnorm_v2" / "best_generators"

latest_ckpt = tf.train.latest_checkpoint(str(resume_dir))
latest_archive = tf.train.latest_checkpoint(str(archive_dir))

print("\n=== COPIED PROJECT CHECK ===")
print("Latest resume checkpoint:", latest_ckpt)
print("Latest archive checkpoint:", latest_archive)
print("Best generator dir exists:", best_dir.exists())

history_file = WORK_ROOT / "results" / "srgan_training_groupnorm_v2.csv"
print("History file exists:", history_file.exists())

if history_file.exists():
    history = pd.read_csv(history_file)
    print("\nLast history row:")
    print(history.tail(1).to_string(index=False))

print("\nSetup copy check finished.")

In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/kaggle/working/project")

DOGS_DATASET_ROOT = Path(
    "/kaggle/input/datasets/blourdhuraju/"
    "dogs-vs-cats-classification/"
    "dogs-vs-cats-classification"
)

split_path = PROJECT_DIR / "data" / "dataset_split.csv"

if not split_path.exists():
    raise FileNotFoundError("Original dataset_split.csv missing!")

split_df = pd.read_csv(split_path)

print("=== ORIGINAL DATASET SPLIT ===")
print("Total images:", len(split_df))
print("Columns:", list(split_df.columns))

print("\nSplit distribution:")
print(split_df["split"].value_counts())

print("\nFirst three CSV rows:")
print(split_df.head(3).to_string(index=False))

# Find available original images
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

available_images = {
    path.relative_to(DOGS_DATASET_ROOT).as_posix()
    for path in DOGS_DATASET_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in image_extensions
}

print("\n=== IMAGE PATH VERIFICATION ===")
print("Image files on Kaggle:", len(available_images))

if "relative_path" in split_df.columns:
    relative_paths = (
        split_df["relative_path"]
        .astype(str)
        .str.replace("\\", "/", regex=False)
        .str.lstrip("/")
    )

    missing = [
        p for p in relative_paths
        if p not in available_images
    ]

    print("Matching images:", len(split_df) - len(missing))
    print("Missing images:", len(missing))

    if missing:
        print("\nFirst missing paths:")
        for p in missing[:5]:
            print(" ", p)
    else:
        print("\nSUCCESS: All original image paths match!")
else:
    print("\nWARNING: No relative_path column found.")

print("\n=== EXAMPLE KAGGLE IMAGE PATHS ===")
for p in sorted(available_images)[:5]:
    print(p)


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/kaggle/working/project")

DOGS_DATASET_ROOT = Path(
    "/kaggle/input/datasets/blourdhuraju/"
    "dogs-vs-cats-classification/"
    "dogs-vs-cats-classification"
)

# Load the original split
split_df = pd.read_csv(
    PROJECT_DIR / "data" / "dataset_split.csv"
)

# Remove the extra folder prefix used in Colab
prefix = "dogs-vs-cats-classification/"

def fix_image_path(relative_path):
    path = str(relative_path).replace("\\", "/")

    if path.startswith(prefix):
        path = path[len(prefix):]

    return path

# Keep original relative_path unchanged
split_df["kaggle_relative_path"] = (
    split_df["relative_path"].apply(fix_image_path)
)

split_df["image_path"] = (
    split_df["kaggle_relative_path"]
    .apply(lambda p: str(DOGS_DATASET_ROOT / p))
)

# Verify all images
missing = [
    path for path in split_df["image_path"]
    if not Path(path).is_file()
]

duplicates = split_df["image_path"].duplicated().sum()

print("=== KAGGLE DATASET VERIFICATION ===")
print("Total images:", len(split_df))
print("Found images:", len(split_df) - len(missing))
print("Missing images:", len(missing))
print("Duplicate paths:", duplicates)

print("\nOriginal split distribution:")
print(split_df["split"].value_counts())

print("\nFirst resolved image:")
print(split_df["image_path"].iloc[0])

if missing:
    print("\nFirst missing paths:")
    for path in missing[:5]:
        print(path)

assert len(split_df) == 24930
assert len(missing) == 0, "Some images are missing!"
assert duplicates == 0, "Duplicate image paths detected!"

print("\nSUCCESS: All 24,930 images correctly mapped!")
print("Original train/validation/test split preserved.")


In [ ]:

from pathlib import Path
from PIL import Image, ImageOps
import pandas as pd
import hashlib
import os
import time

# ============================================================
# PREPARE 128x128 RGB IMAGES FOR SRGAN
# ============================================================

PROJECT_DIR = Path("/kaggle/working/project")
PREPARED_DIR = Path("/kaggle/working/prepared_rgb128")
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (128, 128)
JPEG_QUALITY = 95

# Keep the original train / validation / test split
assert len(split_df) == 24930
assert split_df["image_path"].map(
    lambda p: Path(p).is_file()
).all()

def get_prepared_path(relative_path):
    # Deterministic, collision-resistant filename
    file_id = hashlib.sha256(
        str(relative_path).encode("utf-8")
    ).hexdigest()[:24]

    return PREPARED_DIR / f"{file_id}.jpg"

prepared_df = split_df.copy()

prepared_df["prepared_path"] = (
    prepared_df["relative_path"]
    .apply(lambda p: str(get_prepared_path(p)))
)

start_time = time.perf_counter()
created = 0
reused = 0
errors = []

print("=== PREPARING SRGAN IMAGES ===")
print("Total images:", len(prepared_df))
print("Target size:", IMAGE_SIZE)
print("Format: RGB JPEG")
print("Resizing: Bicubic")
print()

for i, row in enumerate(prepared_df.itertuples(index=False), start=1):

    src = Path(row.image_path)
    dst = Path(row.prepared_path)

    if dst.is_file():
        reused += 1
    else:
        tmp = dst.with_suffix(".part")

        try:
            with Image.open(src) as img:
                img = ImageOps.exif_transpose(img)
                img = img.convert("RGB")
                img = img.resize(
                    IMAGE_SIZE,
                    resample=Image.Resampling.BICUBIC
                )

                img.save(
                    tmp,
                    format="JPEG",
                    quality=JPEG_QUALITY
                )

            os.replace(tmp, dst)
            created += 1

        except Exception as exc:
            errors.append((str(src), str(exc)))
            tmp.unlink(missing_ok=True)

    if i % 2500 == 0 or i == len(prepared_df):
        print(
            f"Processed {i:,}/{len(prepared_df):,} | "
            f"New: {created:,} | "
            f"Reused: {reused:,} | "
            f"Errors: {len(errors)}"
        )

# Verify all images were prepared successfully
missing = [
    p for p in prepared_df["prepared_path"]
    if not Path(p).is_file()
]

if errors or missing:
    print("\nFirst errors:", errors[:5])
    raise RuntimeError(
        f"Preparation incomplete: "
        f"{len(errors)} errors, "
        f"{len(missing)} missing files."
    )

# Re-create our original DataFrames
train_df = prepared_df[
    prepared_df["split"] == "train"
].copy()

val_df = prepared_df[
    prepared_df["split"] == "validation"
].copy()

test_df = prepared_df[
    prepared_df["split"] == "test"
].copy()

assert len(train_df) == 15705
assert len(val_df) == 1746
assert len(test_df) == 7479

# Save Kaggle-specific file mapping
manifest_path = (
    PROJECT_DIR / "data" /
    "prepared_manifest_kaggle.csv"
)

prepared_df.to_csv(manifest_path, index=False)

elapsed = time.perf_counter() - start_time

print("\n=== PREPARATION COMPLETED ===")
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))
print("New files:", created)
print("Reused files:", reused)
print("Missing files:", len(missing))
print(f"Duration: {elapsed / 60:.1f} minutes")
print("Manifest:", manifest_path)
print("\nSUCCESS: SRGAN images ready!")


In [ ]:

from pathlib import Path
from PIL import Image, ImageOps
import pandas as pd
import hashlib
import os
import time

# ============================================================
# KAGGLE - PREPARE 128x128 RGB IMAGES FOR SRGAN
# ============================================================

PROJECT_DIR = Path("/kaggle/working/project")

DOGS_DATASET_ROOT = Path(
    "/kaggle/input/datasets/blourdhuraju/"
    "dogs-vs-cats-classification/"
    "dogs-vs-cats-classification"
)

PREPARED_DIR = Path("/kaggle/working/prepared_rgb128")
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (128, 128)
JPEG_QUALITY = 95

# ------------------------------------------------------------
# 1. Load original dataset split
# ------------------------------------------------------------

split_path = PROJECT_DIR / "data" / "dataset_split.csv"

if not split_path.is_file():
    raise FileNotFoundError(
        f"Original dataset split not found: {split_path}"
    )

split_df = pd.read_csv(split_path)

assert len(split_df) == 24930
assert split_df["relative_path"].is_unique
assert set(split_df["label"].unique()) == {0, 1}

expected_counts = {
    "train": 15705,
    "validation": 1746,
    "test": 7479
}

assert split_df["split"].value_counts().to_dict() == expected_counts

# ------------------------------------------------------------
# 2. Rebuild original image paths for Kaggle
# ------------------------------------------------------------

prefix = "dogs-vs-cats-classification/"

def fix_image_path(relative_path):

    path = str(relative_path).replace("\\", "/")

    if path.startswith(prefix):
        path = path[len(prefix):]

    return path

split_df["kaggle_relative_path"] = (
    split_df["relative_path"].apply(fix_image_path)
)

split_df["image_path"] = (
    split_df["kaggle_relative_path"].apply(
        lambda p: str(DOGS_DATASET_ROOT / p)
    )
)

missing_originals = [
    p for p in split_df["image_path"]
    if not Path(p).is_file()
]

if missing_originals:
    raise FileNotFoundError(
        f"{len(missing_originals)} original images missing. "
        f"First: {missing_originals[0]}"
    )

print("=== ORIGINAL DATASET ===")
print("Total images:", len(split_df))
print("Missing images:", len(missing_originals))
print("\nSplit distribution:")
print(split_df["split"].value_counts())

# ------------------------------------------------------------
# 3. Create deterministic prepared-image filenames
# ------------------------------------------------------------

def get_prepared_path(relative_path):

    file_id = hashlib.sha256(
        str(relative_path).encode("utf-8")
    ).hexdigest()[:24]

    return PREPARED_DIR / f"{file_id}.jpg"

prepared_df = split_df.copy()

prepared_df["prepared_path"] = (
    prepared_df["relative_path"].apply(
        lambda p: str(get_prepared_path(p))
    )
)

assert prepared_df["prepared_path"].is_unique

# ------------------------------------------------------------
# 4. Prepare all images
# ------------------------------------------------------------

start_time = time.perf_counter()

created = 0
reused = 0
errors = []

print("\n=== PREPARING SRGAN IMAGES ===")
print("Total images:", len(prepared_df))
print("Target size:", IMAGE_SIZE)
print("Format: RGB JPEG")
print("Resizing: Bicubic")
print()

for i, row in enumerate(
    prepared_df.itertuples(index=False),
    start=1
):

    src = Path(row.image_path)
    dst = Path(row.prepared_path)

    if dst.is_file() and dst.stat().st_size > 0:
        reused += 1

    else:
        tmp = dst.with_suffix(".part")

        try:

            with Image.open(src) as img:

                img = ImageOps.exif_transpose(img)

                img = img.convert("RGB")

                img = img.resize(
                    IMAGE_SIZE,
                    resample=Image.Resampling.BICUBIC
                )

                img.save(
                    tmp,
                    format="JPEG",
                    quality=JPEG_QUALITY
                )

            os.replace(tmp, dst)

            created += 1

        except Exception as exc:

            errors.append(
                (str(src), str(exc))
            )

            tmp.unlink(missing_ok=True)

    if i % 2500 == 0 or i == len(prepared_df):

        print(
            f"Processed {i:,}/{len(prepared_df):,} | "
            f"New: {created:,} | "
            f"Reused: {reused:,} | "
            f"Errors: {len(errors)}"
        )

# ------------------------------------------------------------
# 5. Verify prepared images
# ------------------------------------------------------------

missing_prepared = [
    p for p in prepared_df["prepared_path"]
    if not Path(p).is_file()
]

if errors or missing_prepared:

    print("\nFirst errors:", errors[:5])

    raise RuntimeError(
        f"Preparation incomplete: "
        f"{len(errors)} errors, "
        f"{len(missing_prepared)} missing images."
    )

# ------------------------------------------------------------
# 6. Rebuild train, validation and test DataFrames
# ------------------------------------------------------------

train_df = prepared_df[
    prepared_df["split"] == "train"
].copy()

val_df = prepared_df[
    prepared_df["split"] == "validation"
].copy()

test_df = prepared_df[
    prepared_df["split"] == "test"
].copy()

assert len(train_df) == 15705
assert len(val_df) == 1746
assert len(test_df) == 7479

# ------------------------------------------------------------
# 7. Save Kaggle-specific image manifest
# ------------------------------------------------------------

manifest_path = (
    PROJECT_DIR / "data" /
    "prepared_manifest_kaggle.csv"
)

prepared_df.to_csv(
    manifest_path,
    index=False
)

elapsed = time.perf_counter() - start_time

# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print("\n=== PREPARATION COMPLETED ===")

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

print("New files:", created)
print("Reused files:", reused)
print("Missing files:", len(missing_prepared))

print(f"Duration: {elapsed / 60:.1f} minutes")
print("Manifest:", manifest_path)

print("\nSUCCESS: SRGAN images ready!")


In [ ]:

from pathlib import Path
import pandas as pd
import tensorflow as tf
import shutil

# ============================================================
# RESTORE KAGGLE ENVIRONMENT AFTER SESSION RESTART
# ============================================================

TRANSFER_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

PROJECT_DIR = Path("/kaggle/working/project")

DOGS_DATASET_ROOT = Path(
    "/kaggle/input/datasets/blourdhuraju/"
    "dogs-vs-cats-classification/"
    "dogs-vs-cats-classification"
)

# Restore project files if the working directory was reset
if not (
    PROJECT_DIR / "data" / "dataset_split.csv"
).exists():

    print("Restoring project files...")

    shutil.copytree(
        TRANSFER_ROOT,
        PROJECT_DIR,
        dirs_exist_ok=True
    )

# Reload original data split
split_df = pd.read_csv(
    PROJECT_DIR / "data" / "dataset_split.csv"
)

# Rebuild Kaggle image paths
prefix = "dogs-vs-cats-classification/"

def fix_image_path(relative_path):
    path = str(relative_path).replace("\\", "/")

    if path.startswith(prefix):
        path = path[len(prefix):]

    return path

split_df["kaggle_relative_path"] = (
    split_df["relative_path"].apply(fix_image_path)
)

split_df["image_path"] = (
    split_df["kaggle_relative_path"]
    .apply(lambda p: str(DOGS_DATASET_ROOT / p))
)

# Verify dataset
assert len(split_df) == 24930

assert split_df["image_path"].map(
    lambda p: Path(p).is_file()
).all()

# Check saved checkpoint
resume_dir = (
    PROJECT_DIR / "checkpoints" /
    "srgan/groupnorm_v2/resume"
)

latest_checkpoint = tf.train.latest_checkpoint(
    str(resume_dir)
)

print("\n=== ENVIRONMENT RESTORED ===")
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Images:", len(split_df))
print("Latest checkpoint:", latest_checkpoint)
print("\nSUCCESS: Environment ready!")


In [ ]:

from pathlib import Path
from PIL import Image, ImageOps
import pandas as pd
import hashlib
import os
import time

# ============================================================
# KAGGLE - PREPARE 128x128 RGB IMAGES FOR SRGAN
# ============================================================

PROJECT_DIR = Path("/kaggle/working/project")

DOGS_DATASET_ROOT = Path(
    "/kaggle/input/datasets/blourdhuraju/"
    "dogs-vs-cats-classification/"
    "dogs-vs-cats-classification"
)

PREPARED_DIR = Path("/kaggle/working/prepared_rgb128")
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (128, 128)
JPEG_QUALITY = 95

# ------------------------------------------------------------
# 1. Load original dataset split
# ------------------------------------------------------------

split_path = PROJECT_DIR / "data" / "dataset_split.csv"

if not split_path.is_file():
    raise FileNotFoundError(
        f"Original dataset split not found: {split_path}"
    )

split_df = pd.read_csv(split_path)

assert len(split_df) == 24930
assert split_df["relative_path"].is_unique
assert set(split_df["label"].unique()) == {0, 1}

expected_counts = {
    "train": 15705,
    "validation": 1746,
    "test": 7479
}

assert split_df["split"].value_counts().to_dict() == expected_counts

# ------------------------------------------------------------
# 2. Rebuild original image paths for Kaggle
# ------------------------------------------------------------

prefix = "dogs-vs-cats-classification/"

def fix_image_path(relative_path):

    path = str(relative_path).replace("\\", "/")

    if path.startswith(prefix):
        path = path[len(prefix):]

    return path

split_df["kaggle_relative_path"] = (
    split_df["relative_path"].apply(fix_image_path)
)

split_df["image_path"] = (
    split_df["kaggle_relative_path"].apply(
        lambda p: str(DOGS_DATASET_ROOT / p)
    )
)

missing_originals = [
    p for p in split_df["image_path"]
    if not Path(p).is_file()
]

if missing_originals:
    raise FileNotFoundError(
        f"{len(missing_originals)} original images missing. "
        f"First: {missing_originals[0]}"
    )

print("=== ORIGINAL DATASET ===")
print("Total images:", len(split_df))
print("Missing images:", len(missing_originals))
print("\nSplit distribution:")
print(split_df["split"].value_counts())

# ------------------------------------------------------------
# 3. Create deterministic prepared-image filenames
# ------------------------------------------------------------

def get_prepared_path(relative_path):

    file_id = hashlib.sha256(
        str(relative_path).encode("utf-8")
    ).hexdigest()[:24]

    return PREPARED_DIR / f"{file_id}.jpg"

prepared_df = split_df.copy()

prepared_df["prepared_path"] = (
    prepared_df["relative_path"].apply(
        lambda p: str(get_prepared_path(p))
    )
)

assert prepared_df["prepared_path"].is_unique

# ------------------------------------------------------------
# 4. Prepare all images
# ------------------------------------------------------------

start_time = time.perf_counter()

created = 0
reused = 0
errors = []

print("\n=== PREPARING SRGAN IMAGES ===")
print("Total images:", len(prepared_df))
print("Target size:", IMAGE_SIZE)
print("Format: RGB JPEG")
print("Resizing: Bicubic")
print()

for i, row in enumerate(
    prepared_df.itertuples(index=False),
    start=1
):

    src = Path(row.image_path)
    dst = Path(row.prepared_path)

    if dst.is_file() and dst.stat().st_size > 0:
        reused += 1

    else:
        tmp = dst.with_suffix(".part")

        try:

            with Image.open(src) as img:

                img = ImageOps.exif_transpose(img)

                img = img.convert("RGB")

                img = img.resize(
                    IMAGE_SIZE,
                    resample=Image.Resampling.BICUBIC
                )

                img.save(
                    tmp,
                    format="JPEG",
                    quality=JPEG_QUALITY
                )

            os.replace(tmp, dst)

            created += 1

        except Exception as exc:

            errors.append(
                (str(src), str(exc))
            )

            tmp.unlink(missing_ok=True)

    if i % 2500 == 0 or i == len(prepared_df):

        print(
            f"Processed {i:,}/{len(prepared_df):,} | "
            f"New: {created:,} | "
            f"Reused: {reused:,} | "
            f"Errors: {len(errors)}"
        )

# ------------------------------------------------------------
# 5. Verify prepared images
# ------------------------------------------------------------

missing_prepared = [
    p for p in prepared_df["prepared_path"]
    if not Path(p).is_file()
]

if errors or missing_prepared:

    print("\nFirst errors:", errors[:5])

    raise RuntimeError(
        f"Preparation incomplete: "
        f"{len(errors)} errors, "
        f"{len(missing_prepared)} missing images."
    )

# ------------------------------------------------------------
# 6. Rebuild train, validation and test DataFrames
# ------------------------------------------------------------

train_df = prepared_df[
    prepared_df["split"] == "train"
].copy()

val_df = prepared_df[
    prepared_df["split"] == "validation"
].copy()

test_df = prepared_df[
    prepared_df["split"] == "test"
].copy()

assert len(train_df) == 15705
assert len(val_df) == 1746
assert len(test_df) == 7479

# ------------------------------------------------------------
# 7. Save Kaggle-specific image manifest
# ------------------------------------------------------------

manifest_path = (
    PROJECT_DIR / "data" /
    "prepared_manifest_kaggle.csv"
)

prepared_df.to_csv(
    manifest_path,
    index=False
)

elapsed = time.perf_counter() - start_time

# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print("\n=== PREPARATION COMPLETED ===")

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

print("New files:", created)
print("Reused files:", reused)
print("Missing files:", len(missing_prepared))

print(f"Duration: {elapsed / 60:.1f} minutes")
print("Manifest:", manifest_path)

print("\nSUCCESS: SRGAN images ready!")


In [ ]:

from pathlib import Path
import tensorflow as tf

PROJECT_DIR = Path("/kaggle/working/project")

generator_path = (
    PROJECT_DIR / "checkpoints" /
    "srgan/groupnorm_v2/best_generators/"
    "generator_best_psnr.keras"
)

print("=== GENERATOR COMPATIBILITY CHECK ===")
print("TensorFlow:", tf.__version__)
print("Model file exists:", generator_path.is_file())

if not generator_path.is_file():
    raise FileNotFoundError(generator_path)

try:
    generator = tf.keras.models.load_model(
        str(generator_path),
        compile=False
    )

    print("\nSUCCESS: Generator loaded!")
    print("Model name:", generator.name)
    print("Input shape:", generator.input_shape)
    print("Output shape:", generator.output_shape)
    print("Parameters:", generator.count_params())

except Exception as error:
    print("\nGenerator loading failed:")
    print(type(error).__name__)
    print(str(error)[:2000])


In [ ]:

import tensorflow as tf
from pathlib import Path

# ============================================================
# RESTORE CUSTOM PIXELSHUFFLE LAYER
# AND LOAD TRAINED SRGAN GENERATOR
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="Midterm"
)
class PixelShuffle(tf.keras.layers.Layer):

    def __init__(self, scale=2, **kwargs):
        super().__init__(**kwargs)
        self.scale = int(scale)

    def call(self, inputs):
        return tf.nn.depth_to_space(
            inputs,
            block_size=self.scale
        )

    def get_config(self):
        config = super().get_config()
        config.update({
            "scale": self.scale
        })
        return config


# ------------------------------------------------------------
# 1. Locate saved generator
# ------------------------------------------------------------

PROJECT_DIR = Path("/kaggle/working/project")

generator_path = (
    PROJECT_DIR
    / "checkpoints"
    / "srgan"
    / "groupnorm_v2"
    / "best_generators"
    / "generator_best_psnr.keras"
)

assert generator_path.is_file(), (
    f"Generator file missing: {generator_path}"
)

# ------------------------------------------------------------
# 2. Load trained generator
# ------------------------------------------------------------

print("=== LOADING TRAINED SRGAN GENERATOR ===")
print("TensorFlow:", tf.__version__)

generator = tf.keras.models.load_model(
    str(generator_path),
    custom_objects={
        "PixelShuffle": PixelShuffle,
        "Midterm>PixelShuffle": PixelShuffle
    },
    compile=False
)

print("\nSUCCESS: Generator loaded!")

print("Model name:", generator.name)
print("Input shape:", generator.input_shape)
print("Output shape:", generator.output_shape)
print("Parameters:", generator.count_params())

# ------------------------------------------------------------
# 3. Test generator with a dummy image
# ------------------------------------------------------------

dummy_input = tf.random.uniform(
    shape=(1, 32, 32, 3),
    dtype=tf.float32
)

dummy_output = generator(
    dummy_input,
    training=False
)

assert tuple(dummy_output.shape) == (
    1, 128, 128, 3
), "Unexpected generator output shape!"

print("\n=== GENERATOR TEST ===")
print("Input:", dummy_input.shape)
print("Output:", dummy_output.shape)
print("Minimum output:", float(tf.reduce_min(dummy_output)))
print("Maximum output:", float(tf.reduce_max(dummy_output)))

print("\nSUCCESS: Generator compatibility test passed!")


In [ ]:

import tensorflow as tf
from tensorflow.keras import layers
from pathlib import Path

# ============================================================
# RESTORE COMPLETE SRGAN GROUPNORM V2 TRAINING STATE
# ============================================================

PROJECT_DIR = Path("/kaggle/working/project")

resume_dir_v2 = (
    PROJECT_DIR / "checkpoints" /
    "srgan" / "groupnorm_v2" / "resume"
)

print("=== SRGAN RECOVERY ===")
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

# ------------------------------------------------------------
# 1. Check previously loaded generator
# ------------------------------------------------------------

assert "generator" in globals(), (
    "First run the PixelShuffle generator loading cell."
)

assert generator.count_params() == 959427
assert tuple(generator.input_shape) == (None, 32, 32, 3)
assert tuple(generator.output_shape) == (None, 128, 128, 3)

print("\nGenerator architecture: OK")

# ------------------------------------------------------------
# 2. Rebuild GroupNorm discriminator
# ------------------------------------------------------------

def build_discriminator():

    inputs = layers.Input(shape=(128, 128, 3))

    # Eight convolutional blocks
    filters_and_strides = [
        (64, 1),
        (64, 2),
        (128, 1),
        (128, 2),
        (256, 1),
        (256, 2),
        (512, 1),
        (512, 2)
    ]

    x = inputs

    for i, (filters, stride) in enumerate(
        filters_and_strides
    ):

        x = layers.Conv2D(
            filters,
            kernel_size=3,
            strides=stride,
            padding="same"
        )(x)

        if i > 0:
            x = layers.GroupNormalization(
                groups=32,
                axis=-1,
                epsilon=1e-5
            )(x)

        x = layers.LeakyReLU(
            negative_slope=0.2
        )(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(256)(x)

    x = layers.LeakyReLU(
        negative_slope=0.2
    )(x)

    outputs = layers.Dense(1)(x)

    return tf.keras.Model(
        inputs,
        outputs,
        name="SRGAN_Discriminator_GroupNorm"
    )


discriminator = build_discriminator()

print("Discriminator parameters:",
      discriminator.count_params())

assert discriminator.count_params() == 4820673, (
    "Discriminator architecture does not match "
    "the original GroupNorm v2."
)

print("Discriminator architecture: OK")

# ------------------------------------------------------------
# 3. Recreate original optimizers
# ------------------------------------------------------------

GENERATOR_LR = 1e-4
DISCRIMINATOR_LR = 1e-4

g_optimizer_v2 = tf.keras.optimizers.Adam(
    learning_rate=GENERATOR_LR,
    beta_1=0.9,
    global_clipnorm=1.0
)

d_optimizer_v2 = tf.keras.optimizers.Adam(
    learning_rate=DISCRIMINATOR_LR,
    beta_1=0.9,
    global_clipnorm=1.0
)

g_optimizer_v2.build(
    generator.trainable_variables
)

d_optimizer_v2.build(
    discriminator.trainable_variables
)

completed_gan_epochs_v2 = tf.Variable(
    0,
    dtype=tf.int64,
    trainable=False
)

# ------------------------------------------------------------
# 4. Construct complete training checkpoint
# ------------------------------------------------------------

checkpoint_v2 = tf.train.Checkpoint(
    generator=generator,
    discriminator=discriminator,
    generator_optimizer=g_optimizer_v2,
    discriminator_optimizer=d_optimizer_v2,
    epoch=completed_gan_epochs_v2
)

latest_checkpoint = tf.train.latest_checkpoint(
    str(resume_dir_v2)
)

if latest_checkpoint is None:
    raise FileNotFoundError(
        "No resume checkpoint found!"
    )

print("\nCheckpoint found:")
print(latest_checkpoint)

# ------------------------------------------------------------
# 5. Restore complete training state
# ------------------------------------------------------------

restore_status = checkpoint_v2.restore(
    latest_checkpoint
)

try:
    restore_status.assert_consumed()
    print("\nSUCCESS: Full checkpoint restored!")
    restore_complete = True

except (AssertionError, ValueError) as error:
    restore_complete = False
    print("\nWARNING: Full checkpoint verification failed.")
    print(str(error)[:3000])

# ------------------------------------------------------------
# 6. Verify restored training progress
# ------------------------------------------------------------

restored_epoch = int(
    completed_gan_epochs_v2.numpy()
)

g_iterations = int(
    g_optimizer_v2.iterations.numpy()
)

d_iterations = int(
    d_optimizer_v2.iterations.numpy()
)

print("\n=== RESTORED TRAINING STATE ===")
print("Completed epochs:", restored_epoch)
print("Next epoch:", restored_epoch + 1)
print("Generator parameters:", generator.count_params())
print("Discriminator parameters:",
      discriminator.count_params())
print("Generator optimizer iterations:", g_iterations)
print("Discriminator optimizer iterations:", d_iterations)
print("Complete restore:", restore_complete)

# Do not continue training unless restoration is verified
if (
    restore_complete
    and restored_epoch == 27
    and g_iterations == 27 * 982
    and d_iterations == 27 * 982
):
    print("\nSUCCESS: Ready to resume at epoch 28!")
else:
    print(
        "\nSTOP: Restore needs investigation "
        "before training."
    )


In [ ]:

# ============================================================
# SRGAN KAGGLE - UNATTENDED TRAINING WITH CLOUD BACKUPS
# Resume from epoch 27, continue toward epoch 150
# ============================================================

import os
import csv
import json
import time
import shutil
import tempfile
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import tensorflow as tf
import kagglehub

# ------------------------------------------------------------
# 1. Configuration and safety checks
# ------------------------------------------------------------

PROJECT_DIR = Path("/kaggle/working/project")

TOTAL_GAN_EPOCHS = 150
RUN_TO_EPOCH = 150

SR_BATCH_SIZE = 16
SEED = 42

BACKUP_INTERVAL = 5
ARCHIVE_INTERVAL = 10

BACKUP_DATASET = (
    "niklasohlenroth/applied-ai-midterm-srgan-checkpoints"
)

assert globals().get("restore_complete", False), (
    "STOP: Run the complete checkpoint restore cell first."
)

assert int(completed_gan_epochs_v2.numpy()) == 27, (
    "STOP: Expected completed epoch 27."
)

assert int(g_optimizer_v2.iterations.numpy()) == 27 * 982
assert int(d_optimizer_v2.iterations.numpy()) == 27 * 982

assert tf.config.list_physical_devices("GPU"), (
    "STOP: No GPU available."
)

assert len(train_df) == 15705
assert len(val_df) == 1746

print("=== SRGAN KAGGLE TRAINING ===")
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Restored epoch:", int(completed_gan_epochs_v2.numpy()))
print("Training target:", RUN_TO_EPOCH)

# ------------------------------------------------------------
# 2. Rebuild original SRGAN input pipeline
# ------------------------------------------------------------

AUTOTUNE = tf.data.AUTOTUNE

def load_sr_pair(image_path, training=False):

    image_bytes = tf.io.read_file(image_path)

    hr = tf.image.decode_jpeg(
        image_bytes,
        channels=3
    )

    hr = tf.image.convert_image_dtype(
        hr,
        tf.float32
    )

    hr = tf.ensure_shape(
        hr,
        (128, 128, 3)
    )

    if training:
        hr = tf.image.random_flip_left_right(hr)

    lr = tf.image.resize(
        hr,
        (32, 32),
        method="bicubic",
        antialias=True
    )

    lr = tf.clip_by_value(lr, 0.0, 1.0)

    return lr, hr


def make_sr_dataset(df, training=False):

    ds = tf.data.Dataset.from_tensor_slices(
        df["prepared_path"].astype(str).tolist()
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(df),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        lambda path: load_sr_pair(
            path,
            training=training
        ),
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(SR_BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)

    return ds


sr_train_ds = make_sr_dataset(
    train_df,
    training=True
)

sr_val_ds = make_sr_dataset(
    val_df,
    training=False
)

assert len(sr_train_ds) == 982
assert len(sr_val_ds) == 110

print("Training batches:", len(sr_train_ds))
print("Validation batches:", len(sr_val_ds))

# ------------------------------------------------------------
# 3. Restore pretrained VGG19 perceptual network
# ------------------------------------------------------------

print("\nLoading pretrained VGG19...")

vgg = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    input_shape=(128, 128, 3)
)

feature_extractor = tf.keras.Model(
    inputs=vgg.input,
    outputs=vgg.get_layer("block3_conv4").output
)

feature_extractor.trainable = False
del vgg

def perceptual_loss(hr_images, sr_images):

    hr_input = tf.keras.applications.vgg19.preprocess_input(
        hr_images * 255.0
    )

    sr_input = tf.keras.applications.vgg19.preprocess_input(
        sr_images * 255.0
    )

    hr_features = feature_extractor(
        hr_input,
        training=False
    )

    sr_features = feature_extractor(
        sr_input,
        training=False
    )

    return tf.reduce_mean(
        tf.square(hr_features - sr_features)
    )

# Original SRGAN v2 loss weights
PIXEL_WEIGHT = 1.0
PERCEPTUAL_WEIGHT = 1e-8
ADVERSARIAL_WEIGHT = 1e-3

bce_loss = tf.keras.losses.BinaryCrossentropy(
    from_logits=True
)

print("VGG19 ready.")

# ------------------------------------------------------------
# 4. Checkpoint and output directories
# ------------------------------------------------------------

gan_dir = (
    PROJECT_DIR / "checkpoints" /
    "srgan" / "groupnorm_v2"
)

resume_dir = gan_dir / "resume"
archive_dir = gan_dir / "archive"
best_dir = gan_dir / "best_generators"

archive_dir.mkdir(parents=True, exist_ok=True)
best_dir.mkdir(parents=True, exist_ok=True)

resume_manager = tf.train.CheckpointManager(
    checkpoint_v2,
    directory=str(resume_dir),
    max_to_keep=3
)

archive_manager = tf.train.CheckpointManager(
    checkpoint_v2,
    directory=str(archive_dir),
    max_to_keep=None
)

history_path = (
    PROJECT_DIR / "results" /
    "srgan_training_groupnorm_v2.csv"
)

tracking_path = (
    best_dir / "best_generator_tracking.json"
)

best_psnr_path = best_dir / "generator_best_psnr.keras"
best_ssim_path = best_dir / "generator_best_ssim.keras"

with open(tracking_path) as f:
    tracking = json.load(f)

best_psnr = float(tracking["best_psnr"]["value"])
best_ssim = float(tracking["best_ssim"]["value"])

# ------------------------------------------------------------
# 5. External backup to existing private Kaggle dataset
# ------------------------------------------------------------

backup_dir = Path(
    "/kaggle/working/srgan_cloud_backup"
)

def save_cloud_backup(epoch_number):

    # Build a portable project snapshot
    if backup_dir.exists():
        shutil.rmtree(backup_dir)

    snapshot = backup_dir / "project"

    snapshot_resume = (
        snapshot / "checkpoints" /
        "srgan/groupnorm_v2/resume"
    )

    snapshot_resume.mkdir(
        parents=True,
        exist_ok=True
    )

    latest = tf.train.latest_checkpoint(
        str(resume_dir)
    )

    if latest is None:
        raise RuntimeError(
            "Cannot upload backup: no checkpoint."
        )

    latest_name = Path(latest).name

    checkpoint_parts = [
        Path(latest + ".index"),
        *Path(latest).parent.glob(
            latest_name + ".data-*"
        )
    ]

    if len(checkpoint_parts) < 2:
        raise RuntimeError(
            "Incomplete checkpoint - backup aborted."
        )

    for part in checkpoint_parts:
        shutil.copy2(
            part,
            snapshot_resume / part.name
        )

    # Portable checkpoint pointer
    (snapshot_resume / "checkpoint").write_text(
        f'model_checkpoint_path: "{latest_name}"\n'
        f'all_model_checkpoint_paths: "{latest_name}"\n'
    )

    # Other essential project files
    important_files = [
        "data/dataset_split.csv",
        "results/srgan_training_groupnorm_v2.csv",
        "checkpoints/model_A/model_A_final.keras",
        "checkpoints/srgan/pretraining/generator_pretrain_best.keras",
        "documentation/srgan_architecture_v2.json",
        "documentation/srgan_training_config.json"
    ]

    for relative in important_files:

        source = PROJECT_DIR / relative

        if source.is_file():
            destination = snapshot / relative
            destination.parent.mkdir(
                parents=True,
                exist_ok=True
            )
            shutil.copy2(source, destination)

    # Best generators and tracking
    shutil.copytree(
        best_dir,
        snapshot / "checkpoints" /
        "srgan/groupnorm_v2/best_generators",
        dirs_exist_ok=True
    )

    # Recovery manifest
    manifest = {
        "platform": "Kaggle",
        "completed_epoch": epoch_number,
        "next_epoch": epoch_number + 1,
        "checkpoint": latest_name,
        "tensorflow_version": tf.__version__,
        "timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

    (snapshot / "recovery_manifest.json").write_text(
        json.dumps(manifest, indent=2)
    )

    print(
        f"\nUploading external backup: epoch {epoch_number}"
    )

    # Creates a new version of the user's existing dataset
    kagglehub.dataset_upload(
        BACKUP_DATASET,
        str(backup_dir),
        version_notes=(
            f"SRGAN Kaggle recovery after epoch {epoch_number}"
        )
    )

    print(
        f"EXTERNAL BACKUP UPLOADED: epoch {epoch_number}"
    )

# Test external backup BEFORE unattended training
current_epoch = int(
    completed_gan_epochs_v2.numpy()
)

print("\nTesting external checkpoint backup...")

save_cloud_backup(current_epoch)

print("\nExternal backup accepted. Training may begin.")

# ------------------------------------------------------------
# 6. Training and validation steps
# ------------------------------------------------------------

@tf.function
def srgan_train_step(lr_images, hr_images):

    # Train discriminator
    generated_for_d = tf.stop_gradient(
        generator(lr_images, training=False)
    )

    n = tf.shape(hr_images)[0]

    mixed_images = tf.concat(
        [hr_images, generated_for_d],
        axis=0
    )

    labels = tf.concat(
        [
            tf.ones((n, 1)),
            tf.zeros((n, 1))
        ],
        axis=0
    )

    indices = tf.random.shuffle(
        tf.range(tf.shape(mixed_images)[0])
    )

    mixed_images = tf.gather(
        mixed_images, indices
    )

    labels = tf.gather(
        labels, indices
    )

    with tf.GradientTape() as d_tape:

        logits = discriminator(
            mixed_images,
            training=True
        )

        d_loss = bce_loss(labels, logits)

    tf.debugging.assert_all_finite(
        d_loss,
        "Discriminator loss is not finite"
    )

    d_grads = d_tape.gradient(
        d_loss,
        discriminator.trainable_variables
    )

    d_optimizer_v2.apply_gradients(
        zip(
            d_grads,
            discriminator.trainable_variables
        )
    )

    # Train generator
    with tf.GradientTape() as g_tape:

        sr = generator(
            lr_images,
            training=True
        )

        fake_logits = discriminator(
            sr,
            training=False
        )

        pixel = tf.reduce_mean(
            tf.square(hr_images - sr)
        )

        feature = perceptual_loss(
            hr_images,
            sr
        )

        adversarial = bce_loss(
            tf.ones_like(fake_logits),
            fake_logits
        )

        weighted_pixel = PIXEL_WEIGHT * pixel
        weighted_feature = PERCEPTUAL_WEIGHT * feature
        weighted_adv = ADVERSARIAL_WEIGHT * adversarial

        g_loss = (
            weighted_pixel +
            weighted_feature +
            weighted_adv
        )

    tf.debugging.assert_all_finite(
        g_loss,
        "Generator loss is not finite"
    )

    g_grads = g_tape.gradient(
        g_loss,
        generator.trainable_variables
    )

    g_optimizer_v2.apply_gradients(
        zip(
            g_grads,
            generator.trainable_variables
        )
    )

    return (
        g_loss,
        d_loss,
        weighted_pixel,
        weighted_feature,
        weighted_adv
    )


@tf.function
def srgan_validate_step(lr_images, hr_images):

    sr = tf.clip_by_value(
        generator(lr_images, training=False),
        0.0,
        1.0
    )

    mse = tf.reduce_mean(
        tf.square(hr_images - sr),
        axis=[1, 2, 3]
    )

    psnr = tf.image.psnr(
        hr_images,
        sr,
        max_val=1.0
    )

    ssim = tf.image.ssim(
        hr_images,
        sr,
        max_val=1.0
    )

    return mse, psnr, ssim

# ------------------------------------------------------------
# 7. Atomic history saving
# ------------------------------------------------------------

history_columns = [
    "epoch",
    "generator_loss",
    "discriminator_loss",
    "pixel_loss",
    "perceptual_loss",
    "adversarial_loss",
    "val_mse",
    "val_psnr",
    "val_ssim",
    "duration_seconds",
    "timestamp_utc"
]

def write_history(row):

    with open(history_path, newline="") as f:
        existing_rows = list(csv.DictReader(f))

    existing_rows = [
        r for r in existing_rows
        if int(r["epoch"]) != row["epoch"]
    ]

    existing_rows.append(row)

    existing_rows.sort(
        key=lambda r: int(r["epoch"])
    )

    temp_path = history_path.with_suffix(".tmp")

    with open(temp_path, "w", newline="") as f:

        writer = csv.DictWriter(
            f,
            fieldnames=history_columns
        )

        writer.writeheader()
        writer.writerows(existing_rows)

    os.replace(temp_path, history_path)

# ------------------------------------------------------------
# 8. Automatic training loop
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("STARTING UNATTENDED SRGAN TRAINING")
print("Completed epochs:", current_epoch)
print("Target:", RUN_TO_EPOCH)
print("Backup every:", BACKUP_INTERVAL, "epochs")
print("=" * 55)

for epoch_number in range(
    current_epoch + 1,
    RUN_TO_EPOCH + 1
):

    start_time = time.perf_counter()

    g_metric = tf.keras.metrics.Mean()
    d_metric = tf.keras.metrics.Mean()

    pixel_metric = tf.keras.metrics.Mean()
    feature_metric = tf.keras.metrics.Mean()
    adv_metric = tf.keras.metrics.Mean()

    print(
        f"\nEpoch {epoch_number}/{TOTAL_GAN_EPOCHS}",
        flush=True
    )

    for step, (lr, hr) in enumerate(
        sr_train_ds,
        start=1
    ):

        g, d, pixel, feature, adv = srgan_train_step(
            lr, hr
        )

        g_metric.update_state(g)
        d_metric.update_state(d)
        pixel_metric.update_state(pixel)
        feature_metric.update_state(feature)
        adv_metric.update_state(adv)

        if step % 100 == 0 or step == len(sr_train_ds):

            print(
                f"Batch {step}/{len(sr_train_ds)} | "
                f"G: {float(g_metric.result()):.5f} | "
                f"D: {float(d_metric.result()):.5f}",
                flush=True
            )

    # Validation
    mse_total = 0.0
    psnr_total = 0.0
    ssim_total = 0.0
    count = 0

    for lr, hr in sr_val_ds:

        mse, psnr, ssim = srgan_validate_step(
            lr, hr
        )

        mse_total += float(tf.reduce_sum(mse))
        psnr_total += float(tf.reduce_sum(psnr))
        ssim_total += float(tf.reduce_sum(ssim))

        count += int(tf.shape(lr)[0])

    val_mse = mse_total / count
    val_psnr = psnr_total / count
    val_ssim = ssim_total / count

    elapsed = time.perf_counter() - start_time

    # Full checkpoint every completed epoch
    completed_gan_epochs_v2.assign(epoch_number)

    saved_checkpoint = resume_manager.save(
        checkpoint_number=epoch_number
    )

    if epoch_number % ARCHIVE_INTERVAL == 0:

        archive_manager.save(
            checkpoint_number=epoch_number
        )

    row = {
        "epoch": epoch_number,
        "generator_loss": float(g_metric.result()),
        "discriminator_loss": float(d_metric.result()),
        "pixel_loss": float(pixel_metric.result()),
        "perceptual_loss": float(feature_metric.result()),
        "adversarial_loss": float(adv_metric.result()),
        "val_mse": val_mse,
        "val_psnr": val_psnr,
        "val_ssim": val_ssim,
        "duration_seconds": round(elapsed, 2),
        "timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

    write_history(row)

    # Best generator exports
    if val_psnr > best_psnr:

        best_psnr = val_psnr
        generator.save(str(best_psnr_path))

        tracking["best_psnr"] = {
            "epoch": epoch_number,
            "value": val_psnr,
            "path": str(best_psnr_path)
        }

        print("New best PSNR generator saved.")

    if val_ssim > best_ssim:

        best_ssim = val_ssim
        generator.save(str(best_ssim_path))

        tracking["best_ssim"] = {
            "epoch": epoch_number,
            "value": val_ssim,
            "path": str(best_ssim_path)
        }

        print("New best SSIM generator saved.")

    tracking["latest_exported_epoch"] = epoch_number

    with open(tracking_path, "w") as f:
        json.dump(tracking, f, indent=2)

    print("\nEpoch summary")
    print("-" * 48)

    print(
        f"G: {row['generator_loss']:.6f} | "
        f"D: {row['discriminator_loss']:.6f}"
    )

    print(f"PSNR: {val_psnr:.3f} dB")
    print(f"SSIM: {val_ssim:.4f}")
    print(f"Duration: {elapsed:.1f} seconds")
    print("Checkpoint:", saved_checkpoint)

    # Persistent backup every 5 epochs
    if epoch_number % BACKUP_INTERVAL == 0:

        save_cloud_backup(epoch_number)

    print("-" * 48, flush=True)

print("\nTRAINING TARGET COMPLETED.")
print("Final epoch:", int(completed_gan_epochs_v2.numpy()))

# Final external backup
save_cloud_backup(
    int(completed_gan_epochs_v2.numpy())
)


In [ ]:

from pathlib import Path
import json
import pandas as pd
import tensorflow as tf

root = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

manifest_path = root / "recovery_manifest.json"
resume_dir = root / "checkpoints/srgan/groupnorm_v2/resume"
history_path = root / "results/srgan_training_groupnorm_v2.csv"

print("=== KAGGLE BACKUP RECOVERY CHECK ===")
print("Dataset found:", root.exists())

if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print("Backup epoch:", manifest.get("completed_epoch"))
else:
    print("No recovery manifest - possibly old dataset version.")

latest = tf.train.latest_checkpoint(str(resume_dir))
print("Checkpoint:", latest)

if history_path.exists():
    history = pd.read_csv(history_path)
    print("History last epoch:", int(history["epoch"].max()))

if latest:
    print("Index exists:", Path(latest + ".index").exists())
    print("Data files:", len(list(resume_dir.glob(
        Path(latest).name + ".data-*"
    ))))

print("=== CHECK COMPLETED ===")


In [ ]:

from pathlib import Path
import tensorflow as tf
import json

print("=== KAGGLE RECOVERY STATUS ===")

# GPU availability
print("\nGPUs:")
print(tf.config.list_physical_devices("GPU"))

# Latest uploaded cloud backup
BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

BACKUP_RESUME = (
    BACKUP_ROOT / "checkpoints" /
    "srgan/groupnorm_v2/resume"
)

print("\nCloud checkpoint:")
print(tf.train.latest_checkpoint(str(BACKUP_RESUME)))

# Existing working checkpoint
WORK_ROOT = Path("/kaggle/working/project")
WORK_RESUME = (
    WORK_ROOT / "checkpoints" /
    "srgan/groupnorm_v2/resume"
)

print("\nWorking checkpoint:")
print(tf.train.latest_checkpoint(str(WORK_RESUME)))

# Check whether prepared images still exist
prepared_dir = Path("/kaggle/working/prepared_rgb128")

if prepared_dir.exists():
    image_count = sum(
        1 for p in prepared_dir.iterdir()
        if p.suffix.lower() == ".jpg"
    )
else:
    image_count = 0

print("\nPrepared images in working directory:")
print(image_count, "/ 24930")

# Look for saved prepared-image folders in Kaggle inputs
print("\nSaved prepared-image folders in Inputs:")

saved_folders = [
    p for p in Path("/kaggle/input").rglob("prepared_rgb128")
    if p.is_dir()
]

if saved_folders:
    for folder in saved_folders:
        print(folder)
else:
    print("No attached prepared-image folder found.")

# Read best model tracking
tracking_path = (
    BACKUP_ROOT / "checkpoints" /
    "srgan/groupnorm_v2/best_generators/"
    "best_generator_tracking.json"
)

if tracking_path.exists():
    with open(tracking_path) as f:
        tracking = json.load(f)

    print("\nBest PSNR:", tracking["best_psnr"])
    print("Best SSIM:", tracking["best_ssim"])

print("\nRecovery inspection completed.")


In [ ]:

from pathlib import Path
from PIL import Image, ImageOps
import tensorflow as tf
import pandas as pd
import hashlib
import shutil
import os
import time

# ============================================================
# KAGGLE RECOVERY - EPOCH 125 + IMAGE PREPARATION
# ============================================================

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

PROJECT_DIR = Path("/kaggle/working/project")

DOGS_DATASET_ROOT = Path(
    "/kaggle/input/datasets/blourdhuraju/"
    "dogs-vs-cats-classification/"
    "dogs-vs-cats-classification"
)

PREPARED_DIR = Path("/kaggle/working/prepared_rgb128")

# ------------------------------------------------------------
# 1. Restore latest cloud backup
# ------------------------------------------------------------

source_resume = (
    BACKUP_ROOT / "checkpoints/srgan/groupnorm_v2/resume"
)

source_ckpt = tf.train.latest_checkpoint(str(source_resume))

assert source_ckpt is not None
assert Path(source_ckpt).name == "ckpt-125", (
    f"Unexpected cloud checkpoint: {source_ckpt}"
)

working_resume = (
    PROJECT_DIR / "checkpoints/srgan/groupnorm_v2/resume"
)

working_ckpt = tf.train.latest_checkpoint(str(working_resume))

def checkpoint_epoch(path):
    return int(Path(path).name.split("-")[-1]) if path else -1

if checkpoint_epoch(working_ckpt) < 125:
    print("Restoring epoch 125 from cloud backup...")
    shutil.copytree(
        BACKUP_ROOT,
        PROJECT_DIR,
        dirs_exist_ok=True
    )
else:
    print("Existing working checkpoint is already current.")

latest = tf.train.latest_checkpoint(str(working_resume))

assert latest is not None
assert checkpoint_epoch(latest) >= 125

print("Restored checkpoint:", latest)

# ------------------------------------------------------------
# 2. Load original dataset split
# ------------------------------------------------------------

split_df = pd.read_csv(
    PROJECT_DIR / "data/dataset_split.csv"
)

assert len(split_df) == 24930

expected_counts = {
    "train": 15705,
    "validation": 1746,
    "test": 7479
}

assert split_df["split"].value_counts().to_dict() == expected_counts

prefix = "dogs-vs-cats-classification/"

def fix_path(path):
    path = str(path).replace("\\", "/")
    if path.startswith(prefix):
        path = path[len(prefix):]
    return path

split_df["image_path"] = (
    split_df["relative_path"]
    .apply(lambda p: str(DOGS_DATASET_ROOT / fix_path(p)))
)

assert split_df["image_path"].map(
    lambda p: Path(p).is_file()
).all()

print("\nOriginal images:", len(split_df))

# ------------------------------------------------------------
# 3. Prepare RGB images
# ------------------------------------------------------------

PREPARED_DIR.mkdir(parents=True, exist_ok=True)

def prepared_path(relative_path):
    name = hashlib.sha256(
        str(relative_path).encode("utf-8")
    ).hexdigest()[:24]
    return PREPARED_DIR / f"{name}.jpg"

prepared_df = split_df.copy()

prepared_df["prepared_path"] = (
    prepared_df["relative_path"]
    .apply(lambda p: str(prepared_path(p)))
)

created = 0
reused = 0
errors = []
start = time.perf_counter()

print("\n=== IMAGE PREPARATION ===")

for i, row in enumerate(prepared_df.itertuples(index=False), 1):

    src = Path(row.image_path)
    dst = Path(row.prepared_path)

    if dst.is_file() and dst.stat().st_size > 0:
        reused += 1
    else:
        tmp = dst.with_suffix(".part")

        try:
            with Image.open(src) as img:
                img = ImageOps.exif_transpose(img)
                img = img.convert("RGB")
                img = img.resize(
                    (128, 128),
                    Image.Resampling.BICUBIC
                )
                img.save(tmp, format="JPEG", quality=95)

            os.replace(tmp, dst)
            created += 1

        except Exception as e:
            errors.append((str(src), str(e)))
            tmp.unlink(missing_ok=True)

    if i % 2500 == 0 or i == len(prepared_df):
        print(
            f"{i:,}/{len(prepared_df):,} | "
            f"New: {created:,} | "
            f"Reused: {reused:,} | "
            f"Errors: {len(errors)}",
            flush=True
        )

assert not errors, f"Preparation errors: {errors[:5]}"

assert all(
    Path(p).is_file()
    for p in prepared_df["prepared_path"]
)

# ------------------------------------------------------------
# 4. Restore train / validation / test DataFrames
# ------------------------------------------------------------

train_df = prepared_df[
    prepared_df["split"] == "train"
].copy()

val_df = prepared_df[
    prepared_df["split"] == "validation"
].copy()

test_df = prepared_df[
    prepared_df["split"] == "test"
].copy()

assert len(train_df) == 15705
assert len(val_df) == 1746
assert len(test_df) == 7479

manifest_path = (
    PROJECT_DIR / "data/prepared_manifest_kaggle.csv"
)

prepared_df.to_csv(manifest_path, index=False)

print("\n=== RECOVERY COMPLETED ===")
print("Checkpoint:", latest)
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))
print("New images:", created)
print("Reused images:", reused)
print("Errors:", len(errors))
print(f"Duration: {(time.perf_counter()-start)/60:.1f} min")
print("\nSUCCESS: Ready for model restoration!")


In [ ]:

import tensorflow as tf
from tensorflow.keras import layers
from pathlib import Path

# ============================================================
# KAGGLE - RESTORE COMPLETE SRGAN FROM EPOCH 125
# ============================================================

PROJECT_DIR = Path("/kaggle/working/project")

restore_complete = False

resume_dir_v2 = (
    PROJECT_DIR / "checkpoints" /
    "srgan" / "groupnorm_v2" / "resume"
)

latest_checkpoint = tf.train.latest_checkpoint(
    str(resume_dir_v2)
)

print("=== SRGAN EPOCH 125 RECOVERY ===")
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Checkpoint:", latest_checkpoint)

assert latest_checkpoint is not None
assert Path(latest_checkpoint).name == "ckpt-125", (
    "STOP: Expected checkpoint 125."
)

assert len(train_df) == 15705
assert len(val_df) == 1746
assert len(test_df) == 7479

# ------------------------------------------------------------
# 1. Register custom PixelShuffle layer
# ------------------------------------------------------------

@tf.keras.utils.register_keras_serializable(
    package="Midterm"
)
class PixelShuffle(tf.keras.layers.Layer):

    def __init__(self, scale=2, **kwargs):
        super().__init__(**kwargs)
        self.scale = int(scale)

    def call(self, inputs):
        return tf.nn.depth_to_space(
            inputs,
            block_size=self.scale
        )

    def get_config(self):
        config = super().get_config()
        config["scale"] = self.scale
        return config

# ------------------------------------------------------------
# 2. Load generator architecture
# ------------------------------------------------------------

generator_path = (
    PROJECT_DIR / "checkpoints" /
    "srgan" / "groupnorm_v2" /
    "best_generators" /
    "generator_best_psnr.keras"
)

assert generator_path.is_file()

generator = tf.keras.models.load_model(
    str(generator_path),
    custom_objects={
        "PixelShuffle": PixelShuffle,
        "Midterm>PixelShuffle": PixelShuffle
    },
    compile=False
)

assert generator.count_params() == 959427

print("\nGenerator architecture restored.")
print("Parameters:", generator.count_params())

# ------------------------------------------------------------
# 3. Rebuild GroupNorm discriminator
# ------------------------------------------------------------

def build_discriminator():

    inputs = layers.Input(shape=(128, 128, 3))

    filters_and_strides = [
        (64, 1),
        (64, 2),
        (128, 1),
        (128, 2),
        (256, 1),
        (256, 2),
        (512, 1),
        (512, 2)
    ]

    x = inputs

    for i, (filters, stride) in enumerate(
        filters_and_strides
    ):

        x = layers.Conv2D(
            filters,
            kernel_size=3,
            strides=stride,
            padding="same"
        )(x)

        if i > 0:
            x = layers.GroupNormalization(
                groups=32,
                axis=-1,
                epsilon=1e-5
            )(x)

        x = layers.LeakyReLU(
            negative_slope=0.2
        )(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256)(x)
    x = layers.LeakyReLU(negative_slope=0.2)(x)

    outputs = layers.Dense(1)(x)

    return tf.keras.Model(
        inputs,
        outputs,
        name="SRGAN_Discriminator_GroupNorm"
    )

discriminator = build_discriminator()

assert discriminator.count_params() == 4820673, (
    "Discriminator architecture mismatch!"
)

print("\nDiscriminator architecture restored.")
print("Parameters:", discriminator.count_params())

# ------------------------------------------------------------
# 4. Rebuild optimizers
# ------------------------------------------------------------

GENERATOR_LR = 1e-4
DISCRIMINATOR_LR = 1e-4

g_optimizer_v2 = tf.keras.optimizers.Adam(
    learning_rate=GENERATOR_LR,
    beta_1=0.9,
    global_clipnorm=1.0
)

d_optimizer_v2 = tf.keras.optimizers.Adam(
    learning_rate=DISCRIMINATOR_LR,
    beta_1=0.9,
    global_clipnorm=1.0
)

g_optimizer_v2.build(
    generator.trainable_variables
)

d_optimizer_v2.build(
    discriminator.trainable_variables
)

completed_gan_epochs_v2 = tf.Variable(
    0,
    dtype=tf.int64,
    trainable=False
)

# ------------------------------------------------------------
# 5. Restore complete checkpoint
# ------------------------------------------------------------

checkpoint_v2 = tf.train.Checkpoint(
    generator=generator,
    discriminator=discriminator,
    generator_optimizer=g_optimizer_v2,
    discriminator_optimizer=d_optimizer_v2,
    epoch=completed_gan_epochs_v2
)

print("\nRestoring complete training state...")

restore_status = checkpoint_v2.restore(
    latest_checkpoint
)

restore_status.assert_consumed()

# ------------------------------------------------------------
# 6. Verify restored training state
# ------------------------------------------------------------

restored_epoch = int(
    completed_gan_epochs_v2.numpy()
)

g_iterations = int(
    g_optimizer_v2.iterations.numpy()
)

d_iterations = int(
    d_optimizer_v2.iterations.numpy()
)

expected_iterations = 125 * 982

print("\n=== RESTORED TRAINING STATE ===")
print("Completed epochs:", restored_epoch)
print("Next epoch:", restored_epoch + 1)
print("Generator optimizer iterations:", g_iterations)
print("Discriminator optimizer iterations:", d_iterations)
print("Expected iterations:", expected_iterations)

assert restored_epoch == 125, (
    "Incorrect restored epoch!"
)

assert g_iterations == expected_iterations, (
    "Generator optimizer state mismatch!"
)

assert d_iterations == expected_iterations, (
    "Discriminator optimizer state mismatch!"
)

restore_complete = True

print("\nSUCCESS: FULL CHECKPOINT RESTORED!")
print("Ready to resume at epoch 126.")


In [ ]:

# ============================================================
# SRGAN GROUPNORM V2 - FINAL TRAINING BLOCK
# Resume from epoch 125 and train until epoch 150
# ============================================================

import os
import csv
import json
import time
import shutil
from pathlib import Path
from datetime import datetime, timezone

import tensorflow as tf
import pandas as pd
import kagglehub

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_DIR = Path("/kaggle/working/project")

TOTAL_GAN_EPOCHS = 150
RUN_TO_EPOCH = 150

SR_BATCH_SIZE = 16
SEED = 42

BACKUP_INTERVAL = 5
ARCHIVE_INTERVAL = 10

BACKUP_DATASET = (
    "niklasohlenroth/applied-ai-midterm-srgan-checkpoints"
)

tf.keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# 2. Verify restored training environment
# ------------------------------------------------------------

assert globals().get("restore_complete", False), (
    "Run the epoch 125 restoration cell first."
)

assert "generator" in globals()
assert "discriminator" in globals()
assert "checkpoint_v2" in globals()
assert "g_optimizer_v2" in globals()
assert "d_optimizer_v2" in globals()

assert len(train_df) == 15705
assert len(val_df) == 1746

assert tf.config.list_physical_devices("GPU"), (
    "No GPU detected."
)

gan_dir = (
    PROJECT_DIR / "checkpoints" /
    "srgan" / "groupnorm_v2"
)

resume_dir = gan_dir / "resume"
archive_dir = gan_dir / "archive"
best_dir = gan_dir / "best_generators"

archive_dir.mkdir(parents=True, exist_ok=True)
best_dir.mkdir(parents=True, exist_ok=True)

history_path = (
    PROJECT_DIR / "results" /
    "srgan_training_groupnorm_v2.csv"
)

tracking_path = best_dir / "best_generator_tracking.json"

best_psnr_path = best_dir / "generator_best_psnr.keras"
best_ssim_path = best_dir / "generator_best_ssim.keras"

assert history_path.is_file()
assert tracking_path.is_file()
assert best_psnr_path.is_file()
assert best_ssim_path.is_file()

# Re-read the latest checkpoint.
# This also protects against restarting this cell after
# an interrupted, partially completed training epoch.

latest_checkpoint = tf.train.latest_checkpoint(
    str(resume_dir)
)

assert latest_checkpoint is not None

checkpoint_v2.restore(
    latest_checkpoint
).assert_consumed()

start_epoch = int(
    completed_gan_epochs_v2.numpy()
)

assert 125 <= start_epoch <= TOTAL_GAN_EPOCHS

print("=== FINAL SRGAN TRAINING ===")
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Checkpoint:", latest_checkpoint)
print("Completed epochs:", start_epoch)
print("Target epoch:", RUN_TO_EPOCH)

# ------------------------------------------------------------
# 3. Restore training history and best models
# ------------------------------------------------------------

history_df = pd.read_csv(history_path)

last_history_epoch = int(
    history_df["epoch"].max()
)

assert last_history_epoch == start_epoch, (
    f"History/checkpoint mismatch: "
    f"history={last_history_epoch}, "
    f"checkpoint={start_epoch}"
)

with open(tracking_path) as f:
    tracking = json.load(f)

best_psnr = float(
    tracking["best_psnr"]["value"]
)

best_ssim = float(
    tracking["best_ssim"]["value"]
)

print("\nBest generator tracking:")
print("Best PSNR:", best_psnr)
print("Best SSIM:", best_ssim)

# ------------------------------------------------------------
# 4. Rebuild the original SRGAN input pipeline
# ------------------------------------------------------------

AUTOTUNE = tf.data.AUTOTUNE

def load_sr_pair(image_path, training=False):

    image_bytes = tf.io.read_file(image_path)

    hr = tf.image.decode_jpeg(
        image_bytes,
        channels=3
    )

    hr = tf.image.convert_image_dtype(
        hr,
        tf.float32
    )

    hr = tf.ensure_shape(
        hr,
        (128, 128, 3)
    )

    if training:
        hr = tf.image.random_flip_left_right(hr)

    lr = tf.image.resize(
        hr,
        (32, 32),
        method="bicubic",
        antialias=True
    )

    lr = tf.clip_by_value(
        lr, 0.0, 1.0
    )

    return lr, hr


def make_sr_dataset(df, training=False):

    paths = (
        df["prepared_path"]
        .astype(str)
        .tolist()
    )

    ds = tf.data.Dataset.from_tensor_slices(paths)

    if training:
        ds = ds.shuffle(
            buffer_size=len(paths),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        lambda path: load_sr_pair(
            path,
            training=training
        ),
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(SR_BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)

    return ds


sr_train_ds = make_sr_dataset(
    train_df,
    training=True
)

sr_val_ds = make_sr_dataset(
    val_df,
    training=False
)

assert len(sr_train_ds) == 982
assert len(sr_val_ds) == 110

print("\nTraining batches:", len(sr_train_ds))
print("Validation batches:", len(sr_val_ds))

expected_iterations = start_epoch * len(sr_train_ds)

assert int(
    g_optimizer_v2.iterations.numpy()
) == expected_iterations

assert int(
    d_optimizer_v2.iterations.numpy()
) == expected_iterations

print("Optimizer states verified.")

# ------------------------------------------------------------
# 5. Restore VGG19 perceptual feature extractor
# ------------------------------------------------------------

print("\nLoading VGG19...")

vgg = tf.keras.applications.VGG19(
    include_top=False,
    weights="imagenet",
    input_shape=(128, 128, 3)
)

feature_extractor = tf.keras.Model(
    inputs=vgg.input,
    outputs=vgg.get_layer("block3_conv4").output
)

feature_extractor.trainable = False

del vgg

def perceptual_loss(hr_images, sr_images):

    hr_input = (
        tf.keras.applications.vgg19.preprocess_input(
            hr_images * 255.0
        )
    )

    sr_input = (
        tf.keras.applications.vgg19.preprocess_input(
            sr_images * 255.0
        )
    )

    hr_features = feature_extractor(
        hr_input,
        training=False
    )

    sr_features = feature_extractor(
        sr_input,
        training=False
    )

    return tf.reduce_mean(
        tf.square(hr_features - sr_features)
    )

PIXEL_WEIGHT = 1.0
PERCEPTUAL_WEIGHT = 1e-8
ADVERSARIAL_WEIGHT = 1e-3

bce_loss = tf.keras.losses.BinaryCrossentropy(
    from_logits=True
)

print("VGG19 ready.")

# ------------------------------------------------------------
# 6. Checkpoint managers
# ------------------------------------------------------------

resume_manager = tf.train.CheckpointManager(
    checkpoint_v2,
    directory=str(resume_dir),
    max_to_keep=3
)

archive_manager = tf.train.CheckpointManager(
    checkpoint_v2,
    directory=str(archive_dir),
    max_to_keep=None
)

# ------------------------------------------------------------
# 7. Save best generators safely
# ------------------------------------------------------------

def save_generator_atomic(model, destination):

    temporary = destination.with_name(
        destination.stem + "_staging.keras"
    )

    model.save(str(temporary))
    os.replace(temporary, destination)


def save_tracking_atomic(data):

    temporary = tracking_path.with_suffix(".tmp")

    with open(temporary, "w") as f:
        json.dump(data, f, indent=2)

    os.replace(temporary, tracking_path)

# ------------------------------------------------------------
# 8. External Kaggle backup
# ------------------------------------------------------------

backup_dir = Path(
    "/kaggle/working/srgan_cloud_backup"
)

def save_cloud_backup(epoch_number):

    print(
        f"\nPreparing external backup: epoch {epoch_number}",
        flush=True
    )

    if backup_dir.exists():
        shutil.rmtree(backup_dir)

    snapshot = backup_dir / "project"

    snapshot_resume = (
        snapshot / "checkpoints" /
        "srgan/groupnorm_v2/resume"
    )

    snapshot_resume.mkdir(
        parents=True,
        exist_ok=True
    )

    latest = tf.train.latest_checkpoint(
        str(resume_dir)
    )

    if latest is None:
        raise RuntimeError(
            "No checkpoint available for backup."
        )

    checkpoint_name = Path(latest).name

    assert checkpoint_name == f"ckpt-{epoch_number}"

    parts = [
        Path(latest + ".index"),
        *Path(latest).parent.glob(
            checkpoint_name + ".data-*"
        )
    ]

    assert len(parts) >= 2
    assert all(p.is_file() for p in parts)

    for part in parts:
        shutil.copy2(
            part,
            snapshot_resume / part.name
        )

    # Portable checkpoint pointer
    (snapshot_resume / "checkpoint").write_text(
        f'model_checkpoint_path: "{checkpoint_name}"\n'
        f'all_model_checkpoint_paths: "{checkpoint_name}"\n'
    )

    # Important project files
    important_files = [
        "data/dataset_split.csv",
        "results/srgan_training_groupnorm_v2.csv",
        "checkpoints/model_A/model_A_final.keras",
        "checkpoints/srgan/pretraining/generator_pretrain_best.keras",
        "documentation/srgan_architecture_v2.json",
        "documentation/srgan_training_config.json"
    ]

    for relative in important_files:

        source = PROJECT_DIR / relative

        if not source.is_file():
            raise FileNotFoundError(source)

        destination = snapshot / relative

        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        shutil.copy2(
            source,
            destination
        )

    # Best generators and metadata
    shutil.copytree(
        best_dir,
        snapshot / "checkpoints" /
        "srgan/groupnorm_v2/best_generators",
        dirs_exist_ok=True
    )

    # Backup metadata
    manifest = {
        "platform": "Kaggle",
        "completed_epoch": epoch_number,
        "next_epoch": epoch_number + 1,
        "checkpoint": checkpoint_name,
        "tensorflow_version": tf.__version__,
        "timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

    (snapshot / "recovery_manifest.json").write_text(
        json.dumps(manifest, indent=2)
    )

    kagglehub.dataset_upload(
        BACKUP_DATASET,
        str(backup_dir),
        version_notes=(
            f"SRGAN recovery checkpoint epoch {epoch_number}"
        )
    )

    print(
        f"EXTERNAL BACKUP UPLOADED: epoch {epoch_number}",
        flush=True
    )
    print(
        "Dataset version may still be processing.",
        flush=True
    )

# ------------------------------------------------------------
# 9. SRGAN training step
# ------------------------------------------------------------

@tf.function
def train_step(lr_images, hr_images):

    # Discriminator update
    generated_for_d = tf.stop_gradient(
        generator(lr_images, training=False)
    )

    batch_size = tf.shape(hr_images)[0]

    combined_images = tf.concat(
        [hr_images, generated_for_d],
        axis=0
    )

    labels = tf.concat(
        [
            tf.ones((batch_size, 1)),
            tf.zeros((batch_size, 1))
        ],
        axis=0
    )

    indices = tf.random.shuffle(
        tf.range(tf.shape(combined_images)[0])
    )

    combined_images = tf.gather(
        combined_images, indices
    )

    labels = tf.gather(
        labels, indices
    )

    with tf.GradientTape() as d_tape:

        d_logits = discriminator(
            combined_images,
            training=True
        )

        d_loss = bce_loss(
            labels,
            d_logits
        )

    tf.debugging.assert_all_finite(
        d_loss,
        "Invalid discriminator loss"
    )

    d_grads = d_tape.gradient(
        d_loss,
        discriminator.trainable_variables
    )

    d_optimizer_v2.apply_gradients(
        zip(
            d_grads,
            discriminator.trainable_variables
        )
    )

    # Generator update
    with tf.GradientTape() as g_tape:

        sr_images = generator(
            lr_images,
            training=True
        )

        fake_logits = discriminator(
            sr_images,
            training=False
        )

        pixel_loss = tf.reduce_mean(
            tf.square(hr_images - sr_images)
        )

        feature_loss = perceptual_loss(
            hr_images,
            sr_images
        )

        adversarial_loss = bce_loss(
            tf.ones_like(fake_logits),
            fake_logits
        )

        weighted_pixel = (
            PIXEL_WEIGHT * pixel_loss
        )

        weighted_feature = (
            PERCEPTUAL_WEIGHT * feature_loss
        )

        weighted_adv = (
            ADVERSARIAL_WEIGHT * adversarial_loss
        )

        g_loss = (
            weighted_pixel
            + weighted_feature
            + weighted_adv
        )

    tf.debugging.assert_all_finite(
        g_loss,
        "Invalid generator loss"
    )

    g_grads = g_tape.gradient(
        g_loss,
        generator.trainable_variables
    )

    g_optimizer_v2.apply_gradients(
        zip(
            g_grads,
            generator.trainable_variables
        )
    )

    return (
        g_loss,
        d_loss,
        weighted_pixel,
        weighted_feature,
        weighted_adv
    )

# ------------------------------------------------------------
# 10. Validation step
# ------------------------------------------------------------

@tf.function
def validate_step(lr_images, hr_images):

    sr_images = generator(
        lr_images,
        training=False
    )

    sr_images = tf.clip_by_value(
        sr_images,
        0.0,
        1.0
    )

    mse = tf.reduce_mean(
        tf.square(hr_images - sr_images),
        axis=[1, 2, 3]
    )

    psnr = tf.image.psnr(
        hr_images,
        sr_images,
        max_val=1.0
    )

    ssim = tf.image.ssim(
        hr_images,
        sr_images,
        max_val=1.0
    )

    return mse, psnr, ssim

# ------------------------------------------------------------
# 11. Save training history
# ------------------------------------------------------------

history_columns = [
    "epoch",
    "generator_loss",
    "discriminator_loss",
    "pixel_loss",
    "perceptual_loss",
    "adversarial_loss",
    "val_mse",
    "val_psnr",
    "val_ssim",
    "duration_seconds",
    "timestamp_utc"
]

def write_history(row):

    with open(history_path, newline="") as f:
        rows = list(csv.DictReader(f))

    rows = [
        old for old in rows
        if int(old["epoch"]) != row["epoch"]
    ]

    rows.append(row)

    rows.sort(
        key=lambda item: int(item["epoch"])
    )

    temp_path = history_path.with_suffix(".tmp")

    with open(temp_path, "w", newline="") as f:

        writer = csv.DictWriter(
            f,
            fieldnames=history_columns
        )

        writer.writeheader()
        writer.writerows(rows)

    os.replace(temp_path, history_path)

# ------------------------------------------------------------
# 12. Final training loop
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("STARTING FINAL SRGAN TRAINING BLOCK")
print("Resume epoch:", start_epoch)
print("Target epoch:", RUN_TO_EPOCH)
print("Backup interval:", BACKUP_INTERVAL)
print("=" * 55)

for epoch_number in range(
    start_epoch + 1,
    RUN_TO_EPOCH + 1
):

    start_time = time.perf_counter()

    metrics = {
        "g": tf.keras.metrics.Mean(),
        "d": tf.keras.metrics.Mean(),
        "pixel": tf.keras.metrics.Mean(),
        "feature": tf.keras.metrics.Mean(),
        "adv": tf.keras.metrics.Mean()
    }

    print(
        f"\nEpoch {epoch_number}/{TOTAL_GAN_EPOCHS}",
        flush=True
    )

    # Training
    for step, (lr_images, hr_images) in enumerate(
        sr_train_ds,
        start=1
    ):

        g, d, pixel, feature, adv = train_step(
            lr_images,
            hr_images
        )

        metrics["g"].update_state(g)
        metrics["d"].update_state(d)

        metrics["pixel"].update_state(pixel)
        metrics["feature"].update_state(feature)
        metrics["adv"].update_state(adv)

        if step % 100 == 0 or step == len(sr_train_ds):

            print(
                f"Batch {step}/{len(sr_train_ds)} | "
                f"G: {float(metrics['g'].result()):.5f} | "
                f"D: {float(metrics['d'].result()):.5f}",
                flush=True
            )

    # Validation
    mse_total = 0.0
    psnr_total = 0.0
    ssim_total = 0.0
    count = 0

    for lr_images, hr_images in sr_val_ds:

        mse, psnr, ssim = validate_step(
            lr_images,
            hr_images
        )

        mse_total += float(tf.reduce_sum(mse))
        psnr_total += float(tf.reduce_sum(psnr))
        ssim_total += float(tf.reduce_sum(ssim))

        count += int(tf.shape(lr_images)[0])

    val_mse = mse_total / count
    val_psnr = psnr_total / count
    val_ssim = ssim_total / count

    elapsed = time.perf_counter() - start_time

    # Save complete training state
    completed_gan_epochs_v2.assign(epoch_number)

    checkpoint_path = resume_manager.save(
        checkpoint_number=epoch_number
    )

    if epoch_number % ARCHIVE_INTERVAL == 0:

        archive_manager.save(
            checkpoint_number=epoch_number
        )

    row = {
        "epoch": epoch_number,
        "generator_loss": float(metrics["g"].result()),
        "discriminator_loss": float(metrics["d"].result()),
        "pixel_loss": float(metrics["pixel"].result()),
        "perceptual_loss": float(metrics["feature"].result()),
        "adversarial_loss": float(metrics["adv"].result()),
        "val_mse": val_mse,
        "val_psnr": val_psnr,
        "val_ssim": val_ssim,
        "duration_seconds": round(elapsed, 2),
        "timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

    write_history(row)

    # Best PSNR generator
    if val_psnr > best_psnr:

        best_psnr = val_psnr

        save_generator_atomic(
            generator,
            best_psnr_path
        )

        tracking["best_psnr"] = {
            "epoch": epoch_number,
            "value": float(val_psnr),
            "path": str(best_psnr_path)
        }

        print("NEW BEST PSNR GENERATOR SAVED")

    # Best SSIM generator
    if val_ssim > best_ssim:

        best_ssim = val_ssim

        save_generator_atomic(
            generator,
            best_ssim_path
        )

        tracking["best_ssim"] = {
            "epoch": epoch_number,
            "value": float(val_ssim),
            "path": str(best_ssim_path)
        }

        print("NEW BEST SSIM GENERATOR SAVED")

    tracking["latest_exported_epoch"] = epoch_number

    save_tracking_atomic(tracking)

    # Summary
    print("\nEpoch summary")
    print("-" * 50)

    print(
        f"Generator Loss: "
        f"{row['generator_loss']:.6f}"
    )

    print(
        f"Discriminator Loss: "
        f"{row['discriminator_loss']:.6f}"
    )

    print(f"Validation PSNR: {val_psnr:.3f} dB")
    print(f"Validation SSIM: {val_ssim:.4f}")
    print(f"Duration: {elapsed:.1f} seconds")
    print("Checkpoint:", checkpoint_path)

    # External backup every five epochs
    if epoch_number % BACKUP_INTERVAL == 0:

        save_cloud_backup(epoch_number)

    print("-" * 50, flush=True)

print("\n" + "=" * 55)
print("SRGAN TRAINING COMPLETED")
print("Final epoch:", int(completed_gan_epochs_v2.numpy()))
print("Best PSNR:", tracking["best_psnr"])
print("Best SSIM:", tracking["best_ssim"])
print("=" * 55)



In [ ]:

from pathlib import Path
import json
import pandas as pd
import tensorflow as tf

# ==========================================
# SRGAN FINAL BACKUP VERIFICATION
# ==========================================

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

manifest_path = BACKUP_ROOT / "recovery_manifest.json"
resume_dir = (
    BACKUP_ROOT / "checkpoints/srgan/groupnorm_v2/resume"
)
history_path = (
    BACKUP_ROOT / "results/srgan_training_groupnorm_v2.csv"
)
tracking_path = (
    BACKUP_ROOT / "checkpoints/srgan/groupnorm_v2/"
    "best_generators/best_generator_tracking.json"
)

print("=== SRGAN FINAL RECOVERY CHECK ===")

if not BACKUP_ROOT.exists():
    raise FileNotFoundError(
        "Checkpoint dataset not attached!"
    )

# Check cloud backup
with open(manifest_path) as f:
    manifest = json.load(f)

print("\nCloud backup epoch:",
      manifest["completed_epoch"])

# Check full checkpoint
latest = tf.train.latest_checkpoint(str(resume_dir))

print("Latest checkpoint:", latest)

if latest:
    index_ok = Path(latest + ".index").is_file()
    data_files = list(
        resume_dir.glob(Path(latest).name + ".data-*")
    )
    print("Checkpoint index:", index_ok)
    print("Checkpoint data files:", len(data_files))

# Check training history
history = pd.read_csv(history_path)

print("\nHistory last epoch:",
      int(history["epoch"].max()))
print("Total logged epochs:", len(history))

print("\nLast 5 epochs:")
print(
    history[
        ["epoch", "generator_loss",
         "discriminator_loss", "val_psnr", "val_ssim"]
    ].tail(5).to_string(index=False)
)

# Check best generators
with open(tracking_path) as f:
    tracking = json.load(f)

print("\n=== BEST GENERATORS ===")
print("Best PSNR:", tracking["best_psnr"])
print("Best SSIM:", tracking["best_ssim"])

if (
    manifest["completed_epoch"] == 150
    and latest is not None
    and Path(latest).name == "ckpt-150"
    and index_ok
    and len(data_files) >= 1
    and int(history["epoch"].max()) == 150
):
    print("\nSUCCESS: SRGAN EPOCH 150 BACKUP VERIFIED!")
else:
    print("\nWARNING: Check latest Kaggle dataset version.")



In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

# ==============================================
# SRGAN - FINAL TRAINING ANALYSIS
# ==============================================

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

FIGURE_DIR = Path(
    "/kaggle/working/srgan_final_figures"
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

history = pd.read_csv(
    BACKUP_ROOT / "results/srgan_training_groupnorm_v2.csv"
).sort_values("epoch")

assert len(history) == 150
assert history["epoch"].tolist() == list(range(1, 151))

# Best validation results
best_psnr_row = history.loc[
    history["val_psnr"].idxmax()
]

best_ssim_row = history.loc[
    history["val_ssim"].idxmax()
]

print("=== FINAL SRGAN RESULTS ===")
print("Total epochs:", len(history))

print(
    f"Best PSNR: {best_psnr_row['val_psnr']:.4f} dB "
    f"(epoch {int(best_psnr_row['epoch'])})"
)

print(
    f"Best SSIM: {best_ssim_row['val_ssim']:.4f} "
    f"(epoch {int(best_ssim_row['epoch'])})"
)

print("\nFinal epoch:")
print(
    history.tail(1)[
        ["epoch", "generator_loss",
         "discriminator_loss", "val_psnr", "val_ssim"]
    ].to_string(index=False)
)

# Confirm the selected generator files exist
best_dir = (
    BACKUP_ROOT / "checkpoints/srgan/"
    "groupnorm_v2/best_generators"
)

for filename in [
    "generator_best_psnr.keras",
    "generator_best_ssim.keras"
]:
    path = best_dir / filename
    assert path.is_file(), f"Missing: {path}"
    print("Verified model:", filename)

# ==============================================
# Figure 1 - Validation PSNR
# ==============================================

plt.figure(figsize=(10, 5))

plt.plot(
    history["epoch"],
    history["val_psnr"],
    label="SRGAN validation PSNR"
)

plt.axhline(
    25.689,
    linestyle="--",
    label="Pretrained generator"
)

plt.axhline(
    25.251,
    linestyle=":",
    label="Bicubic baseline"
)

plt.scatter(
    [best_psnr_row["epoch"]],
    [best_psnr_row["val_psnr"]],
    s=75,
    zorder=5,
    label="Best epoch"
)

plt.xlabel("Adversarial training epoch")
plt.ylabel("PSNR (dB)")
plt.title("SRGAN Validation PSNR")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "srgan_final_psnr.png",
    dpi=250
)
plt.savefig(
    FIGURE_DIR / "srgan_final_psnr.pdf"
)
plt.show()

# ==============================================
# Figure 2 - Validation SSIM
# ==============================================

plt.figure(figsize=(10, 5))

plt.plot(
    history["epoch"],
    history["val_ssim"],
    label="SRGAN validation SSIM"
)

plt.axhline(
    0.7436,
    linestyle="--",
    label="Pretrained generator"
)

plt.axhline(
    0.7164,
    linestyle=":",
    label="Bicubic baseline"
)

plt.scatter(
    [best_ssim_row["epoch"]],
    [best_ssim_row["val_ssim"]],
    s=75,
    zorder=5,
    label="Best epoch"
)

plt.xlabel("Adversarial training epoch")
plt.ylabel("SSIM")
plt.title("SRGAN Validation SSIM")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "srgan_final_ssim.png",
    dpi=250
)
plt.savefig(
    FIGURE_DIR / "srgan_final_ssim.pdf"
)
plt.show()

# ==============================================
# Figure 3 - GAN Losses
# ==============================================

plt.figure(figsize=(10, 5))

plt.plot(
    history["epoch"],
    history["generator_loss"],
    label="Generator loss"
)

plt.plot(
    history["epoch"],
    history["discriminator_loss"],
    label="Discriminator loss"
)

plt.yscale("log")
plt.xlabel("Adversarial training epoch")
plt.ylabel("Loss (log scale)")
plt.title("SRGAN Generator and Discriminator Losses")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "srgan_final_losses.png",
    dpi=250
)
plt.savefig(
    FIGURE_DIR / "srgan_final_losses.pdf"
)
plt.show()

print("\nSUCCESS: Final training figures created.")
print("Saved to:", FIGURE_DIR)


In [ ]:

# ============================================================
# SRGAN FINAL VISUAL COMPARISON
# Validation images only - no full preprocessing required
# ============================================================

import io
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from pathlib import Path
from PIL import Image, ImageOps

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

DATASET_ROOT = Path(
    "/kaggle/input/datasets/blourdhuraju/"
    "dogs-vs-cats-classification/"
    "dogs-vs-cats-classification"
)

FIGURE_DIR = Path(
    "/kaggle/working/srgan_final_figures"
)

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Register custom layer
# ------------------------------------------------------------

@tf.keras.utils.register_keras_serializable(
    package="Midterm"
)
class PixelShuffle(tf.keras.layers.Layer):

    def __init__(self, scale=2, **kwargs):
        super().__init__(**kwargs)
        self.scale = int(scale)

    def call(self, inputs):
        return tf.nn.depth_to_space(
            inputs,
            block_size=self.scale
        )

    def get_config(self):
        config = super().get_config()
        config["scale"] = self.scale
        return config

# ------------------------------------------------------------
# 3. Load pretrained and best generators
# ------------------------------------------------------------

pretrain_path = (
    BACKUP_ROOT / "checkpoints/srgan/"
    "pretraining/generator_pretrain_best.keras"
)

best_path = (
    BACKUP_ROOT / "checkpoints/srgan/"
    "groupnorm_v2/best_generators/"
    "generator_best_psnr.keras"
)

custom_layers = {
    "PixelShuffle": PixelShuffle,
    "Midterm>PixelShuffle": PixelShuffle
}

pretrained_generator = tf.keras.models.load_model(
    str(pretrain_path),
    custom_objects=custom_layers,
    compile=False
)

best_generator = tf.keras.models.load_model(
    str(best_path),
    custom_objects=custom_layers,
    compile=False
)

print("Pretrained generator loaded.")
print("Best SRGAN generator loaded (epoch 107).")

# ------------------------------------------------------------
# 4. Recover final generator from checkpoint 150
# ------------------------------------------------------------

final_generator = tf.keras.models.clone_model(
    best_generator
)

resume_dir = (
    BACKUP_ROOT / "checkpoints/srgan/"
    "groupnorm_v2/resume"
)

final_checkpoint = tf.train.latest_checkpoint(
    str(resume_dir)
)

assert final_checkpoint is not None
assert Path(final_checkpoint).name == "ckpt-150"

status = tf.train.Checkpoint(
    generator=final_generator
).restore(final_checkpoint)

status.assert_existing_objects_matched()

print("Final generator restored (epoch 150).")

# ------------------------------------------------------------
# 5. Load original validation split
# ------------------------------------------------------------

split_df = pd.read_csv(
    BACKUP_ROOT / "data/dataset_split.csv"
)

val_df = split_df[
    split_df["split"] == "validation"
].copy()

assert len(val_df) == 1746

# Reproducible selection: 3 cats and 3 dogs
cat_samples = val_df[
    val_df["label"] == 0
].sample(n=3, random_state=42)

dog_samples = val_df[
    val_df["label"] == 1
].sample(n=3, random_state=42)

samples = pd.concat(
    [cat_samples, dog_samples],
    ignore_index=True
)

prefix = "dogs-vs-cats-classification/"

def resolve_path(relative_path):

    path = str(relative_path).replace("\\", "/")

    if path.startswith(prefix):
        path = path[len(prefix):]

    return DATASET_ROOT / path

# ------------------------------------------------------------
# 6. Reproduce original 128x128 JPEG preprocessing
# ------------------------------------------------------------

hr_images = []

for relative_path in samples["relative_path"]:

    image_path = resolve_path(relative_path)

    with Image.open(image_path) as img:

        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")
        img = img.resize(
            (128, 128),
            Image.Resampling.BICUBIC
        )

        # Match the earlier JPEG quality=95 pipeline
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=95)
        buffer.seek(0)

        with Image.open(buffer) as prepared:
            arr = np.asarray(
                prepared.convert("RGB"),
                dtype=np.float32
            ) / 255.0

    hr_images.append(arr)

hr_images = tf.convert_to_tensor(
    np.stack(hr_images),
    dtype=tf.float32
)

# ------------------------------------------------------------
# 7. Create the same 32x32 low-resolution inputs
# ------------------------------------------------------------

lr_images = tf.image.resize(
    hr_images,
    (32, 32),
    method="bicubic",
    antialias=True
)

lr_images = tf.clip_by_value(
    lr_images, 0.0, 1.0
)

bicubic_images = tf.image.resize(
    lr_images,
    (128, 128),
    method="bicubic"
)

bicubic_images = tf.clip_by_value(
    bicubic_images, 0.0, 1.0
)

# ------------------------------------------------------------
# 8. Generate reconstructed images
# ------------------------------------------------------------

pretrained_images = pretrained_generator(
    lr_images,
    training=False
)

best_images = best_generator(
    lr_images,
    training=False
)

final_images = final_generator(
    lr_images,
    training=False
)

all_outputs = [
    lr_images.numpy(),
    bicubic_images.numpy(),
    pretrained_images.numpy(),
    best_images.numpy(),
    final_images.numpy(),
    hr_images.numpy()
]

column_names = [
    "Low Resolution\n32x32",
    "Bicubic\n128x128",
    "Pretrained\nGenerator",
    "SRGAN\nEpoch 107",
    "SRGAN\nEpoch 150",
    "Original\n128x128"
]

# ------------------------------------------------------------
# 9. Plot and save image comparison
# ------------------------------------------------------------

fig, axes = plt.subplots(
    6, 6,
    figsize=(18, 17)
)

for col, title in enumerate(column_names):
    axes[0, col].set_title(
        title,
        fontsize=12,
        fontweight="bold",
        pad=12
    )

for row in range(6):

    category = (
        "Cat" if samples.iloc[row]["label"] == 0
        else "Dog"
    )

    for col in range(6):

        image = np.clip(
            all_outputs[col][row],
            0.0,
            1.0
        )

        axes[row, col].imshow(
            image,
            interpolation=(
                "nearest" if col == 0 else "none"
            )
        )

        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])

        if col == 0:
            axes[row, col].set_ylabel(
                f"{category} {row + 1}",
                fontsize=11
            )

plt.suptitle(
    "SRGAN Super-Resolution: Validation Image Comparison",
    fontsize=16,
    fontweight="bold",
    y=0.995
)

plt.tight_layout(rect=[0, 0, 1, 0.975])

png_path = (
    FIGURE_DIR / "srgan_final_image_comparison.png"
)

pdf_path = (
    FIGURE_DIR / "srgan_final_image_comparison.pdf"
)

fig.savefig(
    png_path,
    dpi=250,
    bbox_inches="tight"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 10. Save selected examples for reproducibility
# ------------------------------------------------------------

samples.to_csv(
    FIGURE_DIR / "srgan_visual_samples.csv",
    index=False
)

print("\n=== FINAL COMPARISON COMPLETED ===")
print("Validation samples:", len(samples))
print("Best model epoch: 107")
print("Final model epoch: 150")
print("PNG:", png_path)
print("PDF:", pdf_path)
print("Sample manifest saved.")
print("\nSUCCESS!")


In [ ]:

# ============================================================
# CLASSIFIER B - DATA PIPELINE
# Best SRGAN generator: epoch 107
# ============================================================

import io
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from PIL import Image, ImageOps

SEED = 42
BATCH_SIZE = 16

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

DATASET_ROOT = Path(
    "/kaggle/input/datasets/blourdhuraju/"
    "dogs-vs-cats-classification/"
    "dogs-vs-cats-classification"
)

# ------------------------------------------------------------
# 1. Load original dataset split
# ------------------------------------------------------------

df = pd.read_csv(
    BACKUP_ROOT / "data/dataset_split.csv"
)

prefix = "dogs-vs-cats-classification/"

def resolve_path(relative_path):
    path = str(relative_path).replace("\\", "/")
    if path.startswith(prefix):
        path = path[len(prefix):]
    return str(DATASET_ROOT / path)

df["image_path"] = df["relative_path"].apply(
    resolve_path
)

assert len(df) == 24930
assert df["image_path"].map(os.path.isfile).all()

train_b_df = df[df["split"] == "train"].copy()
val_b_df = df[df["split"] == "validation"].copy()
test_b_df = df[df["split"] == "test"].copy()

assert len(train_b_df) == 15705
assert len(val_b_df) == 1746
assert len(test_b_df) == 7479

print("=== CLASSIFIER B DATASET ===")
print("Training:", len(train_b_df))
print("Validation:", len(val_b_df))
print("Reserved test:", len(test_b_df))

# ------------------------------------------------------------
# 2. Register original PixelShuffle layer
# ------------------------------------------------------------

@tf.keras.utils.register_keras_serializable(
    package="Midterm"
)
class PixelShuffle(tf.keras.layers.Layer):

    def __init__(self, scale=2, **kwargs):
        super().__init__(**kwargs)
        self.scale = int(scale)

    def call(self, inputs):
        return tf.nn.depth_to_space(
            inputs, block_size=self.scale
        )

    def get_config(self):
        config = super().get_config()
        config["scale"] = self.scale
        return config

# ------------------------------------------------------------
# 3. Load the selected SRGAN generator
# ------------------------------------------------------------

best_model_path = (
    BACKUP_ROOT /
    "checkpoints/srgan/groupnorm_v2/"
    "best_generators/generator_best_psnr.keras"
)

tracking_path = (
    BACKUP_ROOT /
    "checkpoints/srgan/groupnorm_v2/"
    "best_generators/best_generator_tracking.json"
)

with open(tracking_path) as f:
    tracking = json.load(f)

selected_epoch = int(
    tracking["best_psnr"]["epoch"]
)

assert selected_epoch == 107

sr_generator_b = tf.keras.models.load_model(
    str(best_model_path),
    custom_objects={
        "PixelShuffle": PixelShuffle,
        "Midterm>PixelShuffle": PixelShuffle
    },
    compile=False
)

sr_generator_b.trainable = False

print("\nSelected SRGAN epoch:", selected_epoch)
print("Generator parameters:", sr_generator_b.count_params())

# ------------------------------------------------------------
# 4. Match original JPEG preprocessing
# ------------------------------------------------------------

def prepare_original_image(path_value):
    path = os.fsdecode(path_value.item())

    with Image.open(path) as img:
        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")
        img = img.resize(
            (128, 128),
            Image.Resampling.BICUBIC
        )

        # Reproduce previous JPEG quality 95 preprocessing
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=95)
        buffer.seek(0)

        with Image.open(buffer) as prepared:
            result = np.asarray(
                prepared.convert("RGB"),
                dtype=np.uint8
            ).copy()

    return result

def load_lr_image(path, label):
    hr = tf.numpy_function(
        prepare_original_image,
        [path],
        tf.uint8
    )

    hr.set_shape((128, 128, 3))

    hr = tf.image.convert_image_dtype(
        hr, tf.float32
    )

    lr = tf.image.resize(
        hr,
        (32, 32),
        method="bicubic",
        antialias=True
    )

    lr = tf.clip_by_value(lr, 0.0, 1.0)

    return lr, tf.cast(label, tf.float32)

# ------------------------------------------------------------
# 5. Create streaming datasets
# ------------------------------------------------------------

def make_lr_dataset(dataframe, training=False):
    ds = tf.data.Dataset.from_tensor_slices((
        dataframe["image_path"].tolist(),
        dataframe["label"].astype("float32").tolist()
    ))

    if training:
        ds = ds.shuffle(
            len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_lr_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

train_lr_b_ds = make_lr_dataset(
    train_b_df, training=True
)

val_lr_b_ds = make_lr_dataset(
    val_b_df, training=False
)

# ------------------------------------------------------------
# 6. Generate SR images in batches
# ------------------------------------------------------------

def apply_super_resolution(lr, labels):
    sr = sr_generator_b(lr, training=False)
    sr = tf.clip_by_value(sr, 0.0, 1.0)
    return sr, labels

train_sr_b_ds = train_lr_b_ds.map(
    apply_super_resolution,
    num_parallel_calls=1
).prefetch(tf.data.AUTOTUNE)

val_sr_b_ds = val_lr_b_ds.map(
    apply_super_resolution,
    num_parallel_calls=1
).prefetch(tf.data.AUTOTUNE)

# ------------------------------------------------------------
# 7. Test one batch
# ------------------------------------------------------------

for sr_images, labels in train_sr_b_ds.take(1):
    print("\n=== DATA PIPELINE TEST ===")
    print("SR image batch:", sr_images.shape)
    print("Labels:", labels.shape)
    print("Value range:",
          float(tf.reduce_min(sr_images)),
          float(tf.reduce_max(sr_images)))

print("\nSUCCESS: Classifier B input pipeline ready!")


In [ ]:

# ============================================================
# CLASSIFIER B - FIX IMAGE LOADING AND REBUILD DATASETS
# ============================================================

import io
import os
import numpy as np
import tensorflow as tf

from PIL import Image, ImageOps

# ------------------------------------------------------------
# 1. Correct image loading
# ------------------------------------------------------------

def prepare_original_image_fixed(path_value):

    # TensorFlow may pass bytes or a NumPy scalar
    if isinstance(path_value, np.ndarray):
        path_value = path_value.item()

    path = os.fsdecode(path_value)

    with Image.open(path) as img:

        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")

        img = img.resize(
            (128, 128),
            Image.Resampling.BICUBIC
        )

        # Match original Colab JPEG preprocessing
        buffer = io.BytesIO()

        img.save(
            buffer,
            format="JPEG",
            quality=95
        )

        buffer.seek(0)

        with Image.open(buffer) as prepared:
            result = np.asarray(
                prepared.convert("RGB"),
                dtype=np.uint8
            ).copy()

    return result


# ------------------------------------------------------------
# 2. Convert original image to 32x32 input
# ------------------------------------------------------------

def load_lr_image_fixed(path, label):

    hr = tf.numpy_function(
        prepare_original_image_fixed,
        [path],
        tf.uint8
    )

    hr.set_shape((128, 128, 3))

    hr = tf.image.convert_image_dtype(
        hr,
        tf.float32
    )

    lr = tf.image.resize(
        hr,
        (32, 32),
        method="bicubic",
        antialias=True
    )

    lr = tf.clip_by_value(
        lr, 0.0, 1.0
    )

    return lr, tf.cast(label, tf.float32)


# ------------------------------------------------------------
# 3. Rebuild datasets
# ------------------------------------------------------------

SEED = 42
BATCH_SIZE = 16

def make_lr_dataset_fixed(dataframe, training=False):

    dataset = tf.data.Dataset.from_tensor_slices((
        dataframe["image_path"].astype(str).tolist(),
        dataframe["label"].astype("float32").tolist()
    ))

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_lr_image_fixed,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset


train_lr_b_ds = make_lr_dataset_fixed(
    train_b_df,
    training=True
)

val_lr_b_ds = make_lr_dataset_fixed(
    val_b_df,
    training=False
)


# ------------------------------------------------------------
# 4. Apply trained SRGAN generator
# ------------------------------------------------------------

sr_generator_b.trainable = False

def apply_super_resolution_fixed(lr, labels):

    sr = sr_generator_b(
        lr,
        training=False
    )

    sr = tf.clip_by_value(
        sr, 0.0, 1.0
    )

    return sr, labels


train_sr_b_ds = train_lr_b_ds.map(
    apply_super_resolution_fixed,
    num_parallel_calls=1
).prefetch(1)

val_sr_b_ds = val_lr_b_ds.map(
    apply_super_resolution_fixed,
    num_parallel_calls=1
).prefetch(1)


# ------------------------------------------------------------
# 5. Test first batch
# ------------------------------------------------------------

print("=== CLASSIFIER B PIPELINE TEST ===")

for sr_images, labels in train_sr_b_ds.take(1):

    print("SR image batch:", sr_images.shape)
    print("Labels:", labels.shape)

    print(
        "Value range:",
        float(tf.reduce_min(sr_images)),
        float(tf.reduce_max(sr_images))
    )

    assert tuple(sr_images.shape) == (
        16, 128, 128, 3
    )

    assert tuple(labels.shape) == (16,)

print("\nSUCCESS: Classifier B input pipeline ready!")


In [ ]:

# ============================================================
# CLASSIFIER B - RESTORE ENVIRONMENT AFTER KAGGLE RESTART
# No model training in this cell
# ============================================================

import io
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from PIL import Image, ImageOps

SEED = 42
BATCH_SIZE = 16

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

DATASET_ROOT = Path(
    "/kaggle/input/datasets/blourdhuraju/"
    "dogs-vs-cats-classification/"
    "dogs-vs-cats-classification"
)

PROJECT_DIR = Path("/kaggle/working/project")

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Restore original dataset split
# ------------------------------------------------------------

df = pd.read_csv(
    BACKUP_ROOT / "data/dataset_split.csv"
)

prefix = "dogs-vs-cats-classification/"

def resolve_path(relative_path):
    path = str(relative_path).replace("\\", "/")

    if path.startswith(prefix):
        path = path[len(prefix):]

    return str(DATASET_ROOT / path)

df["image_path"] = df["relative_path"].apply(
    resolve_path
)

assert len(df) == 24930
assert df["image_path"].map(os.path.isfile).all()

train_b_df = df[df["split"] == "train"].copy()
val_b_df = df[df["split"] == "validation"].copy()
test_b_df = df[df["split"] == "test"].copy()

assert len(train_b_df) == 15705
assert len(val_b_df) == 1746
assert len(test_b_df) == 7479

print("=== CLASSIFIER B DATASET ===")
print("Training:", len(train_b_df))
print("Validation:", len(val_b_df))
print("Reserved test:", len(test_b_df))

# ------------------------------------------------------------
# 3. Register custom PixelShuffle layer
# ------------------------------------------------------------

@tf.keras.utils.register_keras_serializable(
    package="Midterm"
)
class PixelShuffle(tf.keras.layers.Layer):

    def __init__(self, scale=2, **kwargs):
        super().__init__(**kwargs)
        self.scale = int(scale)

    def call(self, inputs):
        return tf.nn.depth_to_space(
            inputs,
            block_size=self.scale
        )

    def get_config(self):
        config = super().get_config()
        config["scale"] = self.scale
        return config

# ------------------------------------------------------------
# 4. Restore selected SRGAN generator
# ------------------------------------------------------------

best_dir = (
    BACKUP_ROOT /
    "checkpoints/srgan/groupnorm_v2/best_generators"
)

tracking_path = best_dir / "best_generator_tracking.json"

with open(tracking_path) as f:
    tracking = json.load(f)

selected_epoch = int(
    tracking["best_psnr"]["epoch"]
)

assert selected_epoch == 107

sr_generator_b = tf.keras.models.load_model(
    str(best_dir / "generator_best_psnr.keras"),
    custom_objects={
        "PixelShuffle": PixelShuffle,
        "Midterm>PixelShuffle": PixelShuffle
    },
    compile=False
)

sr_generator_b.trainable = False

assert sr_generator_b.count_params() == 959427

print("\nSRGAN generator restored.")
print("Selected epoch:", selected_epoch)

# ------------------------------------------------------------
# 5. Restore original image preprocessing
# ------------------------------------------------------------

def prepare_original_image(path_value):

    if isinstance(path_value, np.ndarray):
        path_value = path_value.item()

    path = os.fsdecode(path_value)

    with Image.open(path) as img:

        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")

        img = img.resize(
            (128, 128),
            Image.Resampling.BICUBIC
        )

        buffer = io.BytesIO()
        img.save(
            buffer,
            format="JPEG",
            quality=95
        )

        buffer.seek(0)

        with Image.open(buffer) as prepared:
            result = np.asarray(
                prepared.convert("RGB"),
                dtype=np.uint8
            ).copy()

    return result


def load_lr_image(path, label):

    hr = tf.numpy_function(
        prepare_original_image,
        [path],
        tf.uint8
    )

    hr.set_shape((128, 128, 3))

    hr = tf.image.convert_image_dtype(
        hr,
        tf.float32
    )

    lr = tf.image.resize(
        hr,
        (32, 32),
        method="bicubic",
        antialias=True
    )

    lr = tf.clip_by_value(
        lr, 0.0, 1.0
    )

    return lr, tf.cast(label, tf.float32)

# ------------------------------------------------------------
# 6. Rebuild datasets
# ------------------------------------------------------------

def make_lr_dataset(dataframe, training=False):

    ds = tf.data.Dataset.from_tensor_slices((
        dataframe["image_path"].astype(str).tolist(),
        dataframe["label"].astype("float32").tolist()
    ))

    if training:
        ds = ds.shuffle(
            len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_lr_image,
        num_parallel_calls=4
    )

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(1)

    return ds


train_lr_b_ds = make_lr_dataset(
    train_b_df, training=True
)

val_lr_b_ds = make_lr_dataset(
    val_b_df, training=False
)

# ------------------------------------------------------------
# 7. Rebuild SRGAN image datasets
# ------------------------------------------------------------

def apply_super_resolution(lr, labels):

    sr = sr_generator_b(lr, training=False)

    sr = tf.clip_by_value(
        sr, 0.0, 1.0
    )

    return sr, labels


train_sr_b_ds = train_lr_b_ds.map(
    apply_super_resolution,
    num_parallel_calls=1
).prefetch(1)

val_sr_b_ds = val_lr_b_ds.map(
    apply_super_resolution,
    num_parallel_calls=1
).prefetch(1)

# ------------------------------------------------------------
# 8. Verify restored pipeline
# ------------------------------------------------------------

print("\n=== PIPELINE TEST ===")

for images, labels in train_sr_b_ds.take(1):

    print("Images:", images.shape)
    print("Labels:", labels.shape)

    assert tuple(images.shape) == (
        16, 128, 128, 3
    )

    assert tuple(labels.shape) == (16,)

print("\nTensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

print("\nSUCCESS: CLASSIFIER B ENVIRONMENT RESTORED!")


In [ ]:

# ============================================================
# CLASSIFIER B - TRANSFER LEARNING (STAGE 1)
# MobileNetV2 on SRGAN reconstructed images
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers
from pathlib import Path

SEED = 42
tf.keras.utils.set_random_seed(SEED)

PROJECT_DIR = Path("/kaggle/working/project")
B_DIR = PROJECT_DIR / "checkpoints" / "model_B"
B_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Verify input pipeline
# ------------------------------------------------------------

assert len(train_b_df) == 15705
assert len(val_b_df) == 1746
assert len(test_b_df) == 7479

assert "train_sr_b_ds" in globals()
assert "val_sr_b_ds" in globals()

sr_generator_b.trainable = False

# ------------------------------------------------------------
# 2. Training augmentation
# ------------------------------------------------------------

def augment_one_image(image):

    image = tf.image.random_flip_left_right(image)

    image = tf.image.random_brightness(
        image, max_delta=0.08
    )

    image = tf.image.random_contrast(
        image, lower=0.9, upper=1.1
    )

    return tf.clip_by_value(image, 0.0, 1.0)


def augment_sr_batch(images, labels):

    images = tf.map_fn(
        augment_one_image,
        images,
        fn_output_signature=tf.float32
    )

    return images, labels


train_b_aug_ds = train_sr_b_ds.map(
    augment_sr_batch,
    num_parallel_calls=1
).prefetch(1)

# Validation images are never augmented
val_b_eval_ds = val_sr_b_ds

# ------------------------------------------------------------
# 3. Build Classifier B
# ------------------------------------------------------------

backbone_b = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights="imagenet"
)

backbone_b.trainable = False

inputs_b = tf.keras.Input(
    shape=(128, 128, 3),
    name="srgan_image"
)

# Convert pixel values from [0,1] to [-1,1]
x = layers.Rescaling(
    scale=2.0,
    offset=-1.0
)(inputs_b)

x = backbone_b(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs_b = layers.Dense(
    1,
    activation="sigmoid"
)(x)

model_b = tf.keras.Model(
    inputs_b,
    outputs_b,
    name="Classifier_B_SRGAN_MobileNetV2"
)

# ------------------------------------------------------------
# 4. Verify model architecture
# ------------------------------------------------------------

total_params = model_b.count_params()

trainable_params = sum(
    int(tf.keras.backend.count_params(w))
    for w in model_b.trainable_weights
)

print("=== CLASSIFIER B ARCHITECTURE ===")
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

assert total_params == 2259265
assert trainable_params == 1281

# ------------------------------------------------------------
# 5. Compile
# ------------------------------------------------------------

model_b.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.AUC(
            name="auc"
        )
    ]
)

# ------------------------------------------------------------
# 6. Callbacks and checkpoints
# ------------------------------------------------------------

callbacks_b = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(
            B_DIR / "model_B_head_best.keras"
        ),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(
            B_DIR / "model_B_head_epoch_{epoch:02d}.keras"
        ),
        save_best_only=False,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        str(B_DIR / "model_B_head_training.csv")
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

# ------------------------------------------------------------
# 7. Train classification head
# ------------------------------------------------------------

print("\n=== TRAINING CLASSIFIER B - STAGE 1 ===")
print("Train images:", len(train_b_df))
print("Validation images:", len(val_b_df))
print("Test images reserved:", len(test_b_df))
print("SRGAN generator: epoch 107")
print("MobileNetV2 backbone: frozen")
print("Maximum epochs: 10")

history_b_head = model_b.fit(
    train_b_aug_ds,
    validation_data=val_b_eval_ds,
    epochs=10,
    callbacks=callbacks_b,
    verbose=1
)

# ------------------------------------------------------------
# 8. Save trained model
# ------------------------------------------------------------

final_head_path = (
    B_DIR / "model_B_head_final.keras"
)

model_b.save(str(final_head_path))

print("\n=== CLASSIFIER B STAGE 1 COMPLETE ===")
print("Epochs completed:", len(history_b_head.history["loss"]))
print("Best validation accuracy:",
      max(history_b_head.history["val_accuracy"]))
print("Best validation AUC:",
      max(history_b_head.history["val_auc"]))
print("Final model:", final_head_path)
print("Checkpoints:", B_DIR)


In [ ]:

# ============================================================
# CLASSIFIER B - OVERNIGHT FINE-TUNING
#
# 1. Save Stage 1 to private Kaggle dataset
# 2. Cache SRGAN images during the first epoch
# 3. Fine-tune last 30 MobileNetV2 layers
# 4. Save best/latest checkpoints after every epoch
# 5. Upload cloud backup after every epoch
# 6. Stop safely after approximately 5.5 hours
# ============================================================

import os
import json
import time
import shutil
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import tensorflow as tf
import kagglehub

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SEED = 42
MAX_FINETUNE_EPOCHS = 20
MAX_RUN_HOURS = 5.5

job_start = time.monotonic()

SOURCE_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

PROJECT_DIR = Path("/kaggle/working/project")

B_DIR = PROJECT_DIR / "checkpoints" / "model_B"
B_DIR.mkdir(parents=True, exist_ok=True)

UPLOAD_ROOT = Path(
    "/kaggle/working/classifier_b_cloud_upload"
)

UPLOAD_DATASET = (
    "niklasohlenroth/applied-ai-midterm-srgan-checkpoints"
)

HEAD_FINAL = B_DIR / "model_B_head_final.keras"
HEAD_BEST = B_DIR / "model_B_head_best.keras"
HEAD_CSV = B_DIR / "model_B_head_training.csv"

FT_BEST = B_DIR / "model_B_finetune_best.keras"
FT_LATEST = B_DIR / "model_B_finetune_latest.keras"
FT_FINAL = B_DIR / "model_B_finetuned_final.keras"
FT_CSV = B_DIR / "model_B_finetuning.csv"

# ------------------------------------------------------------
# 2. Verify existing Stage 1 and data
# ------------------------------------------------------------

assert tf.config.list_physical_devices("GPU"), (
    "STOP: GPU not available."
)

assert "train_sr_b_ds" in globals(), (
    "STOP: Restore Classifier B data pipeline first."
)

assert "val_sr_b_ds" in globals(), (
    "STOP: Restore Classifier B data pipeline first."
)

assert len(train_b_df) == 15705
assert len(val_b_df) == 1746
assert len(test_b_df) == 7479

for path in [HEAD_FINAL, HEAD_BEST, HEAD_CSV]:
    assert path.is_file(), f"Missing Stage 1 file: {path}"

assert not FT_CSV.exists(), (
    "Fine-tuning history already exists. "
    "Use a recovery procedure instead of restarting."
)

with open(SOURCE_ROOT / "recovery_manifest.json") as f:
    srgan_manifest = json.load(f)

assert int(srgan_manifest["completed_epoch"]) == 150

head_history = pd.read_csv(HEAD_CSV)
assert len(head_history) == 9

print("=== PRE-FLIGHT CHECK ===")
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("SRGAN: 150 epochs completed")
print("Classifier B Stage 1: 9 epochs completed")
print("Stage 1 model: verified")
print("Training images:", len(train_b_df))
print("Validation images:", len(val_b_df))

# ------------------------------------------------------------
# 3. Persistent cloud backup
# ------------------------------------------------------------

def upload_classifier_b_backup(stage, epoch_number):

    print(
        f"\nPreparing cloud backup: {stage}, "
        f"epoch {epoch_number}",
        flush=True
    )

    if UPLOAD_ROOT.exists():
        shutil.rmtree(UPLOAD_ROOT)

    target = UPLOAD_ROOT / "project"

    # Preserve the complete original SRGAN backup.
    shutil.copytree(SOURCE_ROOT, target)

    target_b = target / "checkpoints" / "model_B"
    target_b.mkdir(parents=True, exist_ok=True)

    files_to_copy = [
        HEAD_FINAL,
        HEAD_BEST,
        HEAD_CSV,
        FT_BEST,
        FT_LATEST,
        FT_FINAL,
        FT_CSV
    ]

    for source in files_to_copy:
        if source.is_file():
            shutil.copy2(
                source,
                target_b / source.name
            )

    metadata = {
        "srgan_completed_epochs": 150,
        "srgan_selected_epoch": 107,
        "classifier_b_stage": stage,
        "finetune_completed_epochs": epoch_number,
        "timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

    (target / "classifier_b_recovery.json").write_text(
        json.dumps(metadata, indent=2)
    )

    kagglehub.dataset_upload(
        UPLOAD_DATASET,
        str(UPLOAD_ROOT),
        version_notes=(
            f"Classifier B {stage} - epoch {epoch_number}"
        )
    )

    print(
        f"CLOUD BACKUP UPLOADED: "
        f"{stage}, epoch {epoch_number}",
        flush=True
    )


# IMPORTANT: Save Stage 1 BEFORE any further training.
print("\nSaving completed Stage 1 externally...")

upload_classifier_b_backup(
    stage="stage1_complete",
    epoch_number=0
)

print("\nSTAGE 1 EXTERNAL BACKUP ACCEPTED.")

# ------------------------------------------------------------
# 4. Cache super-resolved images
# ------------------------------------------------------------

print("\nPreparing cached SRGAN datasets...")
print(
    "The first epoch will build the image cache. "
    "Later epochs should run faster."
)

# SRGAN outputs are cached in system RAM.
# Approx. 3.5 GB for training + validation images.
# The cache fills automatically during the first epoch.
train_sr_cache = train_sr_b_ds.cache()
val_sr_cache = val_sr_b_ds.cache()

def augment_image(image):

    image = tf.image.random_flip_left_right(image)

    image = tf.image.random_brightness(
        image,
        max_delta=0.08
    )

    image = tf.image.random_contrast(
        image,
        lower=0.9,
        upper=1.1
    )

    return tf.clip_by_value(image, 0.0, 1.0)


def augment_batch(images, labels):

    images = tf.map_fn(
        augment_image,
        images,
        fn_output_signature=tf.float32
    )

    return images, labels


# Shuffle cached batches on each epoch.
# Augmentation is applied AFTER caching, so it stays random.
train_ft_ds = (
    train_sr_cache
    .shuffle(
        buffer_size=128,
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .map(
        augment_batch,
        num_parallel_calls=2
    )
    .prefetch(2)
)

val_ft_ds = val_sr_cache.prefetch(2)

print("Cached pipelines configured.")

# ------------------------------------------------------------
# 5. Load Stage 1 classifier
# ------------------------------------------------------------

model_b = tf.keras.models.load_model(
    str(HEAD_FINAL),
    compile=False
)

assert model_b.count_params() == 2259265

backbone_candidates = [
    layer
    for layer in model_b.layers
    if isinstance(layer, tf.keras.Model)
    and "mobilenetv2" in layer.name.lower()
]

assert len(backbone_candidates) == 1, (
    "Could not uniquely identify MobileNetV2 backbone."
)

backbone_b = backbone_candidates[0]

# ------------------------------------------------------------
# 6. Fine-tune last 30 layers
# ------------------------------------------------------------

backbone_b.trainable = True

for layer in backbone_b.layers:
    layer.trainable = False

for layer in backbone_b.layers[-30:]:

    # BatchNormalization remains frozen.
    if not isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = True

trainable_layers = [
    layer.name for layer in backbone_b.layers
    if layer.trainable
]

print("\n=== FINE-TUNING CONFIGURATION ===")
print("Backbone:", backbone_b.name)
print("Last layers considered:", 30)
print("Actually trainable backbone layers:",
      len(trainable_layers))
print("BatchNormalization: frozen")
print("Learning rate: 0.00001")

# Same low learning rate as Classifier A fine-tuning.
model_b.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.AUC(
            name="auc"
        )
    ]
)

# ------------------------------------------------------------
# 7. Automatic cloud backup callback
# ------------------------------------------------------------

class CloudBackupCallback(tf.keras.callbacks.Callback):

    def on_epoch_end(self, epoch, logs=None):

        completed_epoch = epoch + 1

        # This callback runs after model checkpoint
        # and CSVLogger, so files are available.
        upload_classifier_b_backup(
            stage="finetuning",
            epoch_number=completed_epoch
        )

        elapsed_hours = (
            time.monotonic() - job_start
        ) / 3600.0

        print(
            f"\nTotal job runtime: "
            f"{elapsed_hours:.2f} hours",
            flush=True
        )

        if elapsed_hours >= MAX_RUN_HOURS:
            print(
                "\nTIME LIMIT REACHED. "
                "Latest checkpoint has been uploaded. "
                "Stopping safely.",
                flush=True
            )
            self.model.stop_training = True


# ------------------------------------------------------------
# 8. Training callbacks
# ------------------------------------------------------------

callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        str(FT_BEST),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        str(FT_LATEST),
        save_best_only=False,
        verbose=0
    ),

    tf.keras.callbacks.CSVLogger(
        str(FT_CSV)
    ),

    CloudBackupCallback(),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

# ------------------------------------------------------------
# 9. Start overnight fine-tuning
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("STARTING CLASSIFIER B OVERNIGHT FINE-TUNING")
print("=" * 55)
print("Maximum epochs:", MAX_FINETUNE_EPOCHS)
print("Early stopping patience:", 5)
print("Maximum planned runtime:", MAX_RUN_HOURS, "hours")
print("Cloud backup: EVERY completed epoch")
print("Generator: SRGAN epoch 107 (frozen)")
print("Test set: untouched")
print("=" * 55)

history_b_finetune = model_b.fit(
    train_ft_ds,
    validation_data=val_ft_ds,
    epochs=MAX_FINETUNE_EPOCHS,
    callbacks=callbacks,
    verbose=2
)

# ------------------------------------------------------------
# 10. Save final selected fine-tuned model
# ------------------------------------------------------------

model_b.save(str(FT_FINAL))

completed_ft_epochs = len(
    history_b_finetune.history["loss"]
)

print("\n=== FINE-TUNING SUMMARY ===")
print("Completed epochs:", completed_ft_epochs)

if completed_ft_epochs > 0:
    print(
        "Best validation accuracy:",
        max(
            history_b_finetune.history["val_accuracy"]
        )
    )
    print(
        "Best validation AUC:",
        max(
            history_b_finetune.history["val_auc"]
        )
    )

print("Final model:", FT_FINAL)

# Final cloud backup
upload_classifier_b_backup(
    stage="finetuning_final",
    epoch_number=completed_ft_epochs
)

print("\n" + "=" * 55)
print("CLASSIFIER B FINE-TUNING FINISHED")
print("Final cloud backup submitted.")
print("=" * 55)


In [ ]:

# ============================================================
# STEP 26 - FINAL TEST EVALUATION: CLASSIFIER A VS B
# Same 7,479 held-out images; no training or augmentation
# Intermediate predictions are saved for safe recovery
# ============================================================

import os
import io
import json
import time
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from PIL import Image, ImageOps

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

SEED = 42
BATCH_SIZE = 16
SAVE_EVERY_BATCHES = 25

PROJECT_DIR = Path("/kaggle/working/project")

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

DATASET_ROOT = Path(
    "/kaggle/input/datasets/blourdhuraju/"
    "dogs-vs-cats-classification/"
    "dogs-vs-cats-classification"
)

RESULTS_DIR = (
    PROJECT_DIR / "results" / "classifier_comparison"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_CSV = (
    RESULTS_DIR / "test_predictions_A_vs_B.csv"
)

METRICS_CSV = (
    RESULTS_DIR / "test_metrics_A_vs_B.csv"
)

METADATA_JSON = (
    RESULTS_DIR / "test_evaluation_metadata.json"
)


def find_model(relative_path):
    candidates = [
        PROJECT_DIR / relative_path,
        BACKUP_ROOT / relative_path
    ]

    for candidate in candidates:
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f"Model not found: {relative_path}\n"
        "Attach the latest private Kaggle checkpoint "
        "dataset or restore your saved session."
    )


MODEL_A_PATH = find_model(
    "checkpoints/model_A/model_A_final.keras"
)

MODEL_B_PATH = find_model(
    "checkpoints/model_B/model_B_finetuned_final.keras"
)

GENERATOR_PATH = find_model(
    "checkpoints/srgan/groupnorm_v2/"
    "best_generators/generator_best_psnr.keras"
)

SPLIT_PATH = find_model(
    "data/dataset_split.csv"
)

print("=== MODEL PATHS ===")
print("Classifier A:", MODEL_A_PATH)
print("Classifier B:", MODEL_B_PATH)
print("Generator:", GENERATOR_PATH)

# ------------------------------------------------------------
# 2. Load reserved test split
# ------------------------------------------------------------

df = pd.read_csv(SPLIT_PATH)

assert len(df) == 24930

test_df = (
    df[df["split"] == "test"]
    .sort_values("relative_path")
    .reset_index(drop=True)
    .copy()
)

assert len(test_df) == 7479
assert not test_df["relative_path"].duplicated().any()
assert set(test_df["label"].unique()) == {0, 1}

def resolve_path(relative_path):
    relative_path = str(relative_path).replace("\\", "/")
    prefix = "dogs-vs-cats-classification/"

    if relative_path.startswith(prefix):
        relative_path = relative_path[len(prefix):]

    return str(DATASET_ROOT / relative_path)

test_df["image_path"] = (
    test_df["relative_path"].apply(resolve_path)
)

assert test_df["image_path"].map(os.path.isfile).all(), (
    "Some source image files are missing."
)

test_paths = test_df["image_path"].tolist()
test_labels = test_df["label"].astype(int).to_numpy()
test_names = test_df["relative_path"].tolist()

print("\nTest images:", len(test_df))
print("Cats:", int((test_labels == 0).sum()))
print("Dogs:", int((test_labels == 1).sum()))

# ------------------------------------------------------------
# 3. Register SRGAN custom layer
# ------------------------------------------------------------

@tf.keras.utils.register_keras_serializable(
    package="Midterm"
)
class PixelShuffle(tf.keras.layers.Layer):

    def __init__(self, scale=2, **kwargs):
        super().__init__(**kwargs)
        self.scale = int(scale)

    def call(self, inputs):
        return tf.nn.depth_to_space(
            inputs,
            block_size=self.scale
        )

    def get_config(self):
        config = super().get_config()
        config["scale"] = self.scale
        return config

# ------------------------------------------------------------
# 4. Load completed models
# ------------------------------------------------------------

print("\nLoading completed models...")

model_a_test = tf.keras.models.load_model(
    str(MODEL_A_PATH),
    compile=False
)

model_b_test = tf.keras.models.load_model(
    str(MODEL_B_PATH),
    compile=False
)

generator_test = tf.keras.models.load_model(
    str(GENERATOR_PATH),
    custom_objects={
        "PixelShuffle": PixelShuffle,
        "Midterm>PixelShuffle": PixelShuffle
    },
    compile=False
)

model_a_test.trainable = False
model_b_test.trainable = False
generator_test.trainable = False

assert model_a_test.count_params() == 2259265
assert model_b_test.count_params() == 2259265
assert generator_test.count_params() == 959427

print("All models loaded successfully.")

# ------------------------------------------------------------
# 5. Same preprocessing as training
# ------------------------------------------------------------

def prepare_original_image_test(path_value):

    if isinstance(path_value, np.ndarray):
        path_value = path_value.item()

    path = os.fsdecode(path_value)

    with Image.open(path) as img:

        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")
        img = img.resize(
            (128, 128),
            Image.Resampling.BICUBIC
        )

        buffer = io.BytesIO()

        img.save(
            buffer,
            format="JPEG",
            quality=95
        )

        buffer.seek(0)

        with Image.open(buffer) as prepared:
            result = np.asarray(
                prepared.convert("RGB"),
                dtype=np.uint8
            ).copy()

    return result


def load_test_image(path, label):

    image = tf.numpy_function(
        prepare_original_image_test,
        [path],
        tf.uint8
    )

    image.set_shape((128, 128, 3))

    image = tf.image.convert_image_dtype(
        image,
        tf.float32
    )

    return image, tf.cast(label, tf.int32)


test_ds = tf.data.Dataset.from_tensor_slices((
    test_paths,
    test_labels.astype("int32")
))

test_ds = (
    test_ds
    .map(load_test_image, num_parallel_calls=4)
    .batch(BATCH_SIZE)
    .prefetch(1)
)

# ------------------------------------------------------------
# 6. Recover intermediate predictions if available
# ------------------------------------------------------------

rows = []

if PREDICTIONS_CSV.exists():

    previous = pd.read_csv(PREDICTIONS_CSV)

    expected_columns = [
        "relative_path",
        "label",
        "p_dog_A",
        "p_dog_B"
    ]

    assert list(previous.columns) == expected_columns

    done = len(previous)

    assert done <= len(test_df)
    assert (
        done % BATCH_SIZE == 0 or done == len(test_df)
    ), "Existing results end in a partial batch."

    assert (
        previous["relative_path"].tolist()
        == test_names[:done]
    ), "Existing prediction order does not match test data."

    assert np.array_equal(
        previous["label"].to_numpy(dtype=int),
        test_labels[:done]
    )

    rows = previous.to_dict("records")

    print("\nRecovered predictions:", done)

else:
    done = 0
    print("\nNo previous predictions: starting from zero.")

# ------------------------------------------------------------
# 7. Predict A and B on identical test images
# ------------------------------------------------------------

def save_predictions():
    temporary = PREDICTIONS_CSV.with_suffix(".tmp")

    pd.DataFrame(
        rows,
        columns=[
            "relative_path",
            "label",
            "p_dog_A",
            "p_dog_B"
        ]
    ).to_csv(temporary, index=False)

    os.replace(temporary, PREDICTIONS_CSV)


start_time = time.monotonic()
total_batches = int(np.ceil(len(test_df) / BATCH_SIZE))
completed_batches = done // BATCH_SIZE

print("\n" + "=" * 55)
print("FINAL HELD-OUT TEST EVALUATION")
print("=" * 55)
print("Already evaluated:", done)
print("Remaining images:", len(test_df) - done)
print("Total batches:", total_batches)
print("Intermediate save: every 25 batches")
print("No training / no test-time augmentation")
print("=" * 55)

if done < len(test_df):

    remaining_ds = test_ds.skip(completed_batches)

    for batch_index, (hr_images, labels) in enumerate(
        remaining_ds,
        start=completed_batches
    ):

        # Classifier A: original 128x128 image
        pred_a = model_a_test(
            hr_images,
            training=False
        ).numpy().reshape(-1)

        # Classifier B: 128 -> 32 -> SRGAN 128
        lr_images = tf.image.resize(
            hr_images,
            (32, 32),
            method="bicubic",
            antialias=True
        )

        lr_images = tf.clip_by_value(
            lr_images,
            0.0,
            1.0
        )

        sr_images = generator_test(
            lr_images,
            training=False
        )

        sr_images = tf.clip_by_value(
            sr_images,
            0.0,
            1.0
        )

        pred_b = model_b_test(
            sr_images,
            training=False
        ).numpy().reshape(-1)

        labels_np = labels.numpy()

        for i in range(len(labels_np)):

            index = len(rows)

            assert int(labels_np[i]) == int(
                test_labels[index]
            )

            rows.append({
                "relative_path": test_names[index],
                "label": int(labels_np[i]),
                "p_dog_A": float(pred_a[i]),
                "p_dog_B": float(pred_b[i])
            })

        current_batch = batch_index + 1

        if (
            current_batch % SAVE_EVERY_BATCHES == 0
            or len(rows) == len(test_df)
        ):

            save_predictions()

            elapsed_min = (
                time.monotonic() - start_time
            ) / 60

            print(
                f"Batch {current_batch}/{total_batches} "
                f"| Images {len(rows)}/{len(test_df)} "
                f"| {elapsed_min:.1f} min "
                f"| Saved",
                flush=True
            )

assert len(rows) == len(test_df)
save_predictions()

# ------------------------------------------------------------
# 8. Calculate final metrics
# ------------------------------------------------------------

pred_df = pd.read_csv(PREDICTIONS_CSV)

y_true = pred_df["label"].to_numpy(dtype=int)

metrics_rows = []

for name, column in [
    ("Classifier A - Original HR", "p_dog_A"),
    ("Classifier B - SRGAN", "p_dog_B")
]:

    y_prob = pred_df[column].to_numpy(dtype=float)
    y_pred = (y_prob >= 0.5).astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    metrics_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "auc": roc_auc_score(y_true, y_prob),
        "TN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "TP": int(cm[1, 1])
    })

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(METRICS_CSV, index=False)

metadata = {
    "dataset": "Dogs vs Cats Classification",
    "test_images": len(test_df),
    "training_images": 15705,
    "validation_images": 1746,
    "labels": {"0": "cat", "1": "dog"},
    "classifier_A": MODEL_A_PATH.name,
    "classifier_B": MODEL_B_PATH.name,
    "generator": GENERATOR_PATH.name,
    "generator_selected_epoch": 107,
    "input_A": "original 128x128",
    "input_B": "bicubic 32x32 to SRGAN 128x128",
    "threshold": 0.5,
    "random_test_augmentation": False
}

METADATA_JSON.write_text(
    json.dumps(metadata, indent=2)
)

print("\n" + "=" * 65)
print("FINAL TEST RESULTS")
print("=" * 65)

print(
    metrics_df[
        ["model", "accuracy", "precision",
         "recall", "f1", "auc"]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)

print("\nSaved files:")
print(PREDICTIONS_CSV)
print(METRICS_CSV)
print(METADATA_JSON)

# ------------------------------------------------------------
# 9. Upload complete backup with evaluation results
# ------------------------------------------------------------

import kagglehub

UPLOAD_ROOT = Path(
    "/kaggle/working/classifier_test_cloud_upload"
)

if UPLOAD_ROOT.exists():
    shutil.rmtree(UPLOAD_ROOT)

TARGET_ROOT = UPLOAD_ROOT / "project"

# Copy previous cloud backup to preserve SRGAN and A.
shutil.copytree(
    BACKUP_ROOT,
    TARGET_ROOT
)

# Ensure all important B checkpoints are included, even
# if the attached Kaggle dataset is an older snapshot.
essential_b = [
    "model_B_head_best.keras",
    "model_B_head_final.keras",
    "model_B_head_training.csv",
    "model_B_finetune_best.keras",
    "model_B_finetune_latest.keras",
    "model_B_finetuned_final.keras",
    "model_B_finetuning.csv"
]

target_b = TARGET_ROOT / "checkpoints/model_B"
target_b.mkdir(parents=True, exist_ok=True)

for filename in essential_b:

    relative = Path("checkpoints/model_B") / filename
    source = find_model(str(relative))

    shutil.copy2(
        source,
        target_b / filename
    )

# Include complete new evaluation results.
shutil.copytree(
    RESULTS_DIR,
    TARGET_ROOT / "results/classifier_comparison",
    dirs_exist_ok=True
)

assert (
    TARGET_ROOT /
    "checkpoints/srgan/groupnorm_v2/resume/"
    "ckpt-150.index"
).is_file()

assert (
    TARGET_ROOT /
    "checkpoints/model_B/model_B_finetuned_final.keras"
).is_file()

print("\nUploading final test results to private cloud dataset...")

try:

    kagglehub.dataset_upload(
        "niklasohlenroth/applied-ai-midterm-srgan-checkpoints",
        str(UPLOAD_ROOT),
        version_notes=(
            "Completed Classifier A vs B held-out test evaluation"
        )
    )

    print("\nFINAL TEST CLOUD BACKUP UPLOADED.")

except Exception as error:

    print("\nCLOUD UPLOAD FAILED:")
    print(repr(error))
    print("Results remain saved locally at:", RESULTS_DIR)
    print("Do not reset the session before backing them up.")

print("\n" + "=" * 65)
print("CLASSIFIER A VS B TEST EVALUATION COMPLETE")
print("=" * 65)


In [ ]:

# ============================================================
# STEP 27 - FINAL FIGURES AND STATISTICAL COMPARISON
# No training; uses existing held-out test predictions
# ============================================================

import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from pathlib import Path
from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix
)

PROJECT_DIR = Path("/kaggle/working/project")

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

REL_RESULTS = Path("results/classifier_comparison")
RESULTS_DIR = PROJECT_DIR / REL_RESULTS

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PRED_FILE = "test_predictions_A_vs_B.csv"

pred_path = RESULTS_DIR / PRED_FILE

if not pred_path.exists():
    pred_path = BACKUP_ROOT / REL_RESULTS / PRED_FILE

assert pred_path.exists(), "Test predictions not found."

df = pd.read_csv(pred_path)

assert len(df) == 7479
assert not df["relative_path"].duplicated().any()
assert not df[
    ["label", "p_dog_A", "p_dog_B"]
].isna().any().any()

y_true = df["label"].to_numpy(dtype=int)

p_a = df["p_dog_A"].to_numpy(dtype=float)
p_b = df["p_dog_B"].to_numpy(dtype=float)

assert set(np.unique(y_true)) == {0, 1}
assert np.all((p_a >= 0) & (p_a <= 1))
assert np.all((p_b >= 0) & (p_b <= 1))

y_a = (p_a >= 0.5).astype(int)
y_b = (p_b >= 0.5).astype(int)

# ------------------------------------------------------------
# 1. Recalculate metrics from saved predictions
# ------------------------------------------------------------

metric_names = [
    "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"
]

def calculate_metrics(probabilities, predictions):

    return np.array([
        accuracy_score(y_true, predictions),
        precision_score(y_true, predictions),
        recall_score(y_true, predictions),
        f1_score(y_true, predictions),
        roc_auc_score(y_true, probabilities)
    ])

metrics_a = calculate_metrics(p_a, y_a)
metrics_b = calculate_metrics(p_b, y_b)

print("=== VERIFIED TEST RESULTS ===")

for name, a, b in zip(
    metric_names, metrics_a, metrics_b
):
    print(
        f"{name:12s} "
        f"A={a:.5f}  B={b:.5f}  "
        f"Difference={a-b:+.5f}"
    )

# ------------------------------------------------------------
# 2. Paired statistical comparison
# ------------------------------------------------------------

correct_a = (y_a == y_true)
correct_b = (y_b == y_true)

both_correct = int(
    np.sum(correct_a & correct_b)
)

a_only = int(
    np.sum(correct_a & ~correct_b)
)

b_only = int(
    np.sum(~correct_a & correct_b)
)

both_wrong = int(
    np.sum(~correct_a & ~correct_b)
)

assert (
    both_correct + a_only + b_only + both_wrong
) == 7479

# Exact McNemar test:
# compare cases correctly predicted by only one model.
discordant = a_only + b_only

if discordant > 0:
    mcnemar_p = binomtest(
        min(a_only, b_only),
        discordant,
        p=0.5,
        alternative="two-sided"
    ).pvalue
else:
    mcnemar_p = 1.0

# Paired bootstrap confidence interval for accuracy difference.
rng = np.random.default_rng(42)

paired_differences = (
    correct_a.astype(np.int8)
    - correct_b.astype(np.int8)
)

bootstrap_differences = []

N_BOOTSTRAP = 3000
N = len(df)

# Process in chunks to avoid large memory allocations.
for start in range(0, N_BOOTSTRAP, 100):

    count = min(
        100,
        N_BOOTSTRAP - start
    )

    indices = rng.integers(
        0, N,
        size=(count, N)
    )

    bootstrap_differences.extend(
        paired_differences[indices]
        .mean(axis=1)
        .tolist()
    )

ci_low, ci_high = np.quantile(
    bootstrap_differences,
    [0.025, 0.975]
)

accuracy_difference = (
    metrics_a[0] - metrics_b[0]
)

statistics = {
    "test_images": N,
    "both_correct": both_correct,
    "A_correct_B_wrong": a_only,
    "A_wrong_B_correct": b_only,
    "both_wrong": both_wrong,
    "accuracy_A": float(metrics_a[0]),
    "accuracy_B": float(metrics_b[0]),
    "accuracy_difference": float(
        accuracy_difference
    ),
    "accuracy_difference_95ci": [
        float(ci_low),
        float(ci_high)
    ],
    "bootstrap_resamples": N_BOOTSTRAP,
    "mcnemar_exact_p": float(mcnemar_p),
    "threshold": 0.5
}

stats_path = RESULTS_DIR / "paired_statistics.json"

stats_path.write_text(
    json.dumps(statistics, indent=2)
)

print("\n=== PAIRED STATISTICAL ANALYSIS ===")
print("Both correct:", both_correct)
print("Only A correct:", a_only)
print("Only B correct:", b_only)
print("Both wrong:", both_wrong)

print(
    "Accuracy difference: "
    f"{100*accuracy_difference:.3f} percentage points"
)

print(
    "95% paired bootstrap CI: "
    f"[{100*ci_low:.3f}, {100*ci_high:.3f}] "
    "percentage points"
)

print(f"Exact McNemar p-value: {mcnemar_p:.6g}")

# ------------------------------------------------------------
# 3. Figure folders and formatting
# ------------------------------------------------------------

VECTOR_DIR = RESULTS_DIR / "figures" / "vector"
RASTER_DIR = RESULTS_DIR / "figures" / "raster"

VECTOR_DIR.mkdir(parents=True, exist_ok=True)
RASTER_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "figure.dpi": 120,
    "savefig.dpi": 250,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

def save_figure(fig, name):

    fig.savefig(
        VECTOR_DIR / f"{name}.pdf",
        bbox_inches="tight"
    )

    fig.savefig(
        VECTOR_DIR / f"{name}.svg",
        bbox_inches="tight"
    )

    fig.savefig(
        RASTER_DIR / f"{name}.png",
        dpi=250,
        bbox_inches="tight"
    )

    plt.close(fig)

# ------------------------------------------------------------
# 4. Figure 1 - Model performance
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 4.3))

positions = np.arange(len(metric_names))

ax.scatter(
    metrics_a * 100,
    positions - 0.12,
    s=65,
    label="Classifier A - Original HR",
    marker="o"
)

ax.scatter(
    metrics_b * 100,
    positions + 0.12,
    s=65,
    label="Classifier B - SRGAN",
    marker="s"
)

for i in range(len(metric_names)):
    ax.plot(
        [metrics_b[i]*100, metrics_a[i]*100],
        [positions[i], positions[i]],
        linewidth=1,
        alpha=0.4
    )

ax.set_yticks(positions)
ax.set_yticklabels(metric_names)
ax.invert_yaxis()

ax.set_xlim(88, 100.5)
ax.set_xlabel("Test performance (%)")
ax.set_title(
    "Classifier A vs. B - Held-out Test Performance\n"
    "(Zoomed horizontal axis: 88-100%)"
)

ax.grid(axis="x", alpha=0.25)
ax.legend(loc="lower right", fontsize=8)

fig.tight_layout()
save_figure(fig, "test_performance_comparison")

# ------------------------------------------------------------
# 5. Figure 2 - ROC curves
# ------------------------------------------------------------

fpr_a, tpr_a, _ = roc_curve(y_true, p_a)
fpr_b, tpr_b, _ = roc_curve(y_true, p_b)

fig, ax = plt.subplots(figsize=(6, 5))

ax.plot(
    fpr_a,
    tpr_a,
    linewidth=2,
    label=f"Classifier A (AUC={metrics_a[4]:.4f})"
)

ax.plot(
    fpr_b,
    tpr_b,
    linewidth=2,
    label=f"Classifier B (AUC={metrics_b[4]:.4f})"
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1,
    alpha=0.5
)

ax.set(
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
    title="ROC Curves - Held-out Test Set",
    xlim=(0, 1),
    ylim=(0, 1.01)
)

ax.legend(loc="lower right")
ax.grid(alpha=0.2)

fig.tight_layout()
save_figure(fig, "test_roc_curves")

# ------------------------------------------------------------
# 6. Figure 3 - Confusion matrices
# ------------------------------------------------------------

cm_a = confusion_matrix(
    y_true, y_a, labels=[0, 1]
)

cm_b = confusion_matrix(
    y_true, y_b, labels=[0, 1]
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(9, 4)
)

max_count = max(
    cm_a.max(),
    cm_b.max()
)

for ax, cm, title in zip(
    axes,
    [cm_a, cm_b],
    ["Classifier A - Original HR",
     "Classifier B - SRGAN"]
):

    ax.imshow(
        cm,
        cmap="Blues",
        vmin=0,
        vmax=max_count
    )

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(["Cat", "Dog"])
    ax.set_yticklabels(["Cat", "Dog"])

    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(title)

    for row in range(2):
        for col in range(2):

            value = int(cm[row, col])

            ax.text(
                col,
                row,
                f"{value:,}",
                ha="center",
                va="center",
                fontsize=12,
                color=(
                    "white"
                    if value > max_count * 0.55
                    else "black"
                )
            )

fig.suptitle(
    "Confusion Matrices - 7,479 Test Images"
)

fig.tight_layout()
save_figure(fig, "test_confusion_matrices")

# ------------------------------------------------------------
# 7. Figure 4 - Paired correctness
# ------------------------------------------------------------

categories = [
    "Both correct",
    "Only A correct",
    "Only B correct",
    "Both wrong"
]

counts = [
    both_correct,
    a_only,
    b_only,
    both_wrong
]

fig, ax = plt.subplots(figsize=(7.2, 4))

bars = ax.barh(
    categories,
    counts
)

ax.invert_yaxis()

ax.set_xlabel("Number of test images")
ax.set_title(
    "Paired Prediction Agreement - Classifiers A and B"
)

for bar, count in zip(bars, counts):

    ax.text(
        bar.get_width() + 18,
        bar.get_y() + bar.get_height()/2,
        f"{count:,}",
        va="center",
        fontsize=9
    )

ax.set_xlim(0, max(counts)*1.13)
ax.grid(axis="x", alpha=0.2)

fig.tight_layout()
save_figure(fig, "paired_prediction_agreement")

# ------------------------------------------------------------
# 8. Save consolidated numerical results
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "metric": metric_names,
    "classifier_A": metrics_a,
    "classifier_B": metrics_b,
    "difference_A_minus_B": (
        metrics_a - metrics_b
    )
})

comparison.to_csv(
    RESULTS_DIR / "final_metric_comparison.csv",
    index=False
)

print("\n=== FIGURES CREATED ===")

for path in sorted(VECTOR_DIR.glob("*.pdf")):
    print(path.name)

print("\nAll figures also saved as SVG and PNG.")

# ------------------------------------------------------------
# 9. Cloud backup - preserve all previous checkpoints
# ------------------------------------------------------------

UPLOAD_ROOT = Path(
    "/kaggle/working/classifier_figures_cloud_upload"
)

TARGET_ROOT = UPLOAD_ROOT / "project"

required_model = (
    BACKUP_ROOT /
    "checkpoints/model_B/model_B_finetuned_final.keras"
)

assert required_model.is_file(), (
    "Attach the latest checkpoint dataset "
    "before uploading a new dataset version."
)

if UPLOAD_ROOT.exists():
    shutil.rmtree(UPLOAD_ROOT)

shutil.copytree(
    BACKUP_ROOT,
    TARGET_ROOT
)

shutil.copytree(
    RESULTS_DIR,
    TARGET_ROOT / REL_RESULTS,
    dirs_exist_ok=True
)

print("\nUploading scientific figures and statistics...")

try:

    kagglehub.dataset_upload(
        "niklasohlenroth/applied-ai-midterm-srgan-checkpoints",
        str(UPLOAD_ROOT),
        version_notes=(
            "Scientific figures, paired statistics and final "
            "classifier comparison"
        )
    )

    print("\nCLOUD BACKUP UPLOADED.")

except Exception as error:

    print("\nCloud upload failed:", repr(error))
    print("All results are still stored locally.")

print("\n" + "=" * 55)
print("STEP 27 COMPLETE")
print("=" * 55)


In [ ]:

# ============================================================
# STEP 28 - SRGAN IMAGE QUALITY EVALUATION
# 256 held-out test images: 128 cats + 128 dogs
# Compare Bicubic and SRGAN against original HR images
# Save qualitative examples and PSNR/SSIM results
# ============================================================

import io
import os
import json
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import kagglehub

from pathlib import Path
from PIL import Image, ImageOps

SEED = 42

PROJECT_DIR = Path("/kaggle/working/project")

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

DATASET_ROOT = Path(
    "/kaggle/input/datasets/blourdhuraju/"
    "dogs-vs-cats-classification/"
    "dogs-vs-cats-classification"
)

RESULTS_DIR = PROJECT_DIR / "results" / "srgan_quality"
VECTOR_DIR = RESULTS_DIR / "figures" / "vector"
RASTER_DIR = RESULTS_DIR / "figures" / "raster"

for folder in [RESULTS_DIR, VECTOR_DIR, RASTER_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load original test split
# ------------------------------------------------------------

split_path = BACKUP_ROOT / "data/dataset_split.csv"

if not split_path.exists():
    split_path = PROJECT_DIR / "data/dataset_split.csv"

df = pd.read_csv(split_path)

test_df = (
    df[df["split"] == "test"]
    .sort_values("relative_path")
    .reset_index(drop=True)
)

assert len(test_df) == 7479

def resolve_path(relative_path):

    relative_path = str(relative_path).replace("\\", "/")

    prefix = "dogs-vs-cats-classification/"

    if relative_path.startswith(prefix):
        relative_path = relative_path[len(prefix):]

    return DATASET_ROOT / relative_path

test_df["image_path"] = test_df[
    "relative_path"
].apply(resolve_path)

# Stratified, deterministic random sample.
rng = np.random.default_rng(SEED)

cat_ids = np.where(test_df["label"].to_numpy() == 0)[0]
dog_ids = np.where(test_df["label"].to_numpy() == 1)[0]

selected_indices = np.concatenate([
    rng.choice(cat_ids, size=128, replace=False),
    rng.choice(dog_ids, size=128, replace=False)
])

sample_df = (
    test_df.iloc[selected_indices]
    .reset_index(drop=True)
    .copy()
)

assert len(sample_df) == 256

sample_df[
    ["relative_path", "label"]
].to_csv(
    RESULTS_DIR / "quality_sample_manifest.csv",
    index=False
)

print("=== SRGAN QUALITY EVALUATION ===")
print("Selected test images:", len(sample_df))
print("Cats:", int((sample_df.label == 0).sum()))
print("Dogs:", int((sample_df.label == 1).sum()))

# ------------------------------------------------------------
# 2. Load generator
# ------------------------------------------------------------

@tf.keras.utils.register_keras_serializable(
    package="Midterm"
)
class PixelShuffle(tf.keras.layers.Layer):

    def __init__(self, scale=2, **kwargs):
        super().__init__(**kwargs)
        self.scale = int(scale)

    def call(self, inputs):
        return tf.nn.depth_to_space(
            inputs,
            block_size=self.scale
        )

    def get_config(self):
        config = super().get_config()
        config["scale"] = self.scale
        return config


generator_path = (
    BACKUP_ROOT /
    "checkpoints/srgan/groupnorm_v2/"
    "best_generators/generator_best_psnr.keras"
)

if not generator_path.exists():
    generator_path = (
        PROJECT_DIR /
        "checkpoints/srgan/groupnorm_v2/"
        "best_generators/generator_best_psnr.keras"
    )

generator_quality = tf.keras.models.load_model(
    str(generator_path),
    custom_objects={
        "PixelShuffle": PixelShuffle,
        "Midterm>PixelShuffle": PixelShuffle
    },
    compile=False
)

generator_quality.trainable = False

assert generator_quality.count_params() == 959427

# ------------------------------------------------------------
# 3. Match original preprocessing
# ------------------------------------------------------------

def load_hr_image(path):

    with Image.open(path) as img:

        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")

        img = img.resize(
            (128, 128),
            Image.Resampling.BICUBIC
        )

        buffer = io.BytesIO()

        img.save(
            buffer,
            format="JPEG",
            quality=95
        )

        buffer.seek(0)

        with Image.open(buffer) as prepared:

            result = np.asarray(
                prepared.convert("RGB"),
                dtype=np.uint8
            ).copy()

    return result.astype(np.float32) / 255.0

# ------------------------------------------------------------
# 4. Evaluate all 256 images
# ------------------------------------------------------------

quality_rows = []
visual_examples = {}

# Four cats and four dogs from the random sample.
visual_indices = [
    0, 128,
    1, 129,
    2, 130,
    3, 131
]

for start in range(0, len(sample_df), 16):

    end = min(start + 16, len(sample_df))

    batch_rows = sample_df.iloc[start:end]

    hr_np = np.stack([
        load_hr_image(path)
        for path in batch_rows["image_path"]
    ])

    hr = tf.convert_to_tensor(
        hr_np,
        dtype=tf.float32
    )

    # Original HR -> LR: 128x128 -> 32x32
    lr = tf.image.resize(
        hr,
        (32, 32),
        method="bicubic",
        antialias=True
    )

    lr = tf.clip_by_value(lr, 0.0, 1.0)

    # Bicubic baseline
    bicubic = tf.image.resize(
        lr,
        (128, 128),
        method="bicubic"
    )

    bicubic = tf.clip_by_value(
        bicubic, 0.0, 1.0
    )

    # Trained SRGAN generator
    srgan = generator_quality(
        lr,
        training=False
    )

    srgan = tf.clip_by_value(
        srgan, 0.0, 1.0
    )

    # Image quality metrics
    bicubic_psnr = tf.image.psnr(
        hr, bicubic, max_val=1.0
    ).numpy()

    srgan_psnr = tf.image.psnr(
        hr, srgan, max_val=1.0
    ).numpy()

    bicubic_ssim = tf.image.ssim(
        hr, bicubic, max_val=1.0
    ).numpy()

    srgan_ssim = tf.image.ssim(
        hr, srgan, max_val=1.0
    ).numpy()

    # Nearest-neighbor display reveals LR pixel grid.
    lr_display = tf.image.resize(
        lr,
        (128, 128),
        method="nearest"
    ).numpy()

    bicubic_np = bicubic.numpy()
    srgan_np = srgan.numpy()

    for j, (_, record) in enumerate(
        batch_rows.iterrows()
    ):

        global_idx = start + j

        quality_rows.append({
            "relative_path": record["relative_path"],
            "label": int(record["label"]),
            "bicubic_psnr": float(bicubic_psnr[j]),
            "srgan_psnr": float(srgan_psnr[j]),
            "bicubic_ssim": float(bicubic_ssim[j]),
            "srgan_ssim": float(srgan_ssim[j])
        })

        if global_idx in visual_indices:

            visual_examples[global_idx] = {
                "hr": hr_np[j],
                "lr": lr_display[j],
                "bicubic": bicubic_np[j],
                "srgan": srgan_np[j],
                "label": int(record["label"])
            }

    print(
        f"Processed {end}/{len(sample_df)} images",
        flush=True
    )

quality_df = pd.DataFrame(quality_rows)

quality_df.to_csv(
    RESULTS_DIR / "quality_per_image.csv",
    index=False
)

# ------------------------------------------------------------
# 5. Summary and paired confidence intervals
# ------------------------------------------------------------

summary = {}

for metric in ["psnr", "ssim"]:

    baseline = quality_df[
        f"bicubic_{metric}"
    ].to_numpy()

    enhanced = quality_df[
        f"srgan_{metric}"
    ].to_numpy()

    differences = enhanced - baseline

    # Paired bootstrap over the 256 sampled images.
    bootstrap_rng = np.random.default_rng(42)

    indices = bootstrap_rng.integers(
        0,
        len(differences),
        size=(2000, len(differences))
    )

    bootstrap_means = differences[indices].mean(
        axis=1
    )

    ci = np.quantile(
        bootstrap_means,
        [0.025, 0.975]
    )

    summary[metric] = {
        "bicubic_mean": float(baseline.mean()),
        "srgan_mean": float(enhanced.mean()),
        "mean_difference_srgan_minus_bicubic": float(
            differences.mean()
        ),
        "difference_95ci": [
            float(ci[0]),
            float(ci[1])
        ],
        "fraction_srgan_better": float(
            np.mean(differences > 0)
        )
    }

summary["sample_size"] = 256
summary["sample_seed"] = 42
summary["selected_generator_epoch"] = 107
summary["test_split_used_for_generator_selection"] = False

(RESULTS_DIR / "quality_summary.json").write_text(
    json.dumps(summary, indent=2)
)

print("\n=== QUALITY RESULTS ===")

for metric in ["psnr", "ssim"]:

    values = summary[metric]

    print(
        f"{metric.upper()}: "
        f"Bicubic={values['bicubic_mean']:.4f}, "
        f"SRGAN={values['srgan_mean']:.4f}, "
        f"Difference={values['mean_difference_srgan_minus_bicubic']:+.4f}"
    )

# ------------------------------------------------------------
# 6. Save figures
# ------------------------------------------------------------

plt.rcParams.update({
    "font.size": 10,
    "figure.dpi": 120,
    "savefig.dpi": 250,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

def save_figure(fig, name):

    fig.savefig(
        VECTOR_DIR / f"{name}.pdf",
        bbox_inches="tight"
    )

    fig.savefig(
        VECTOR_DIR / f"{name}.svg",
        bbox_inches="tight"
    )

    fig.savefig(
        RASTER_DIR / f"{name}.png",
        dpi=250,
        bbox_inches="tight"
    )

    plt.close(fig)

# Figure 1: Eight representative random examples.
fig, axes = plt.subplots(
    8, 4,
    figsize=(12, 20)
)

column_titles = [
    "Original HR\n128 x 128",
    "Low Resolution\n32 x 32 (nearest display)",
    "Bicubic Upscaling\n128 x 128",
    "SRGAN Upscaling\n128 x 128"
]

for row_idx, image_idx in enumerate(visual_indices):

    example = visual_examples[image_idx]

    label = (
        "Cat" if example["label"] == 0
        else "Dog"
    )

    images = [
        example["hr"],
        example["lr"],
        example["bicubic"],
        example["srgan"]
    ]

    for col_idx, image in enumerate(images):

        ax = axes[row_idx, col_idx]

        ax.imshow(
            np.clip(image, 0, 1)
        )

        ax.set_xticks([])
        ax.set_yticks([])

        if row_idx == 0:
            ax.set_title(
                column_titles[col_idx],
                fontsize=10
            )

        if col_idx == 0:
            ax.set_ylabel(
                f"{label} {row_idx + 1}",
                fontsize=10
            )

fig.suptitle(
    "Held-out Test Examples: Original vs Bicubic vs SRGAN",
    fontsize=13,
    y=0.998
)

fig.tight_layout()
save_figure(fig, "srgan_qualitative_examples")

# Figure 2: PSNR and SSIM means with paired comparison.
fig, axes = plt.subplots(
    1, 2,
    figsize=(9, 4)
)

for ax, metric, unit in zip(
    axes,
    ["psnr", "ssim"],
    ["dB", ""]
):

    values = [
        summary[metric]["bicubic_mean"],
        summary[metric]["srgan_mean"]
    ]

    bars = ax.bar(
        ["Bicubic", "SRGAN"],
        values
    )

    ax.set_title(metric.upper())
    ax.set_ylabel(unit if unit else "SSIM")

    ax.set_ylim(
        0,
        max(values) * 1.15
    )

    for bar, value in zip(bars, values):

        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height(),
            f"{value:.3f}",
            ha="center",
            va="bottom"
        )

    ax.grid(axis="y", alpha=0.2)

fig.suptitle(
    "Image Quality on 256 Held-out Test Images",
    fontsize=12
)

fig.tight_layout()
save_figure(fig, "bicubic_vs_srgan_quality")

print("\nScientific figures saved as PDF, SVG and PNG.")

# ------------------------------------------------------------
# 7. Upload without losing Step 27 results
# ------------------------------------------------------------

STEP27_SNAPSHOT = Path(
    "/kaggle/working/"
    "classifier_figures_cloud_upload/project"
)

if (
    STEP27_SNAPSHOT /
    "results/classifier_comparison/paired_statistics.json"
).is_file():

    source_snapshot = STEP27_SNAPSHOT

else:
    source_snapshot = BACKUP_ROOT

# Verify that the previous results are preserved.
required_previous = [
    "results/classifier_comparison/paired_statistics.json",
    "results/classifier_comparison/figures/vector/"
    "test_roc_curves.pdf",
    "checkpoints/model_B/model_B_finetuned_final.keras",
    "checkpoints/srgan/groupnorm_v2/resume/ckpt-150.index"
]

for relative in required_previous:

    if not (
        source_snapshot / relative
    ).is_file():

        raise FileNotFoundError(
            "Previous backup is incomplete: "
            + relative
            + ". Do not overwrite the cloud dataset."
        )

UPLOAD_ROOT = Path(
    "/kaggle/working/"
    "srgan_quality_cloud_upload"
)

if UPLOAD_ROOT.exists():
    shutil.rmtree(UPLOAD_ROOT)

TARGET_ROOT = UPLOAD_ROOT / "project"

shutil.copytree(
    source_snapshot,
    TARGET_ROOT
)

shutil.copytree(
    RESULTS_DIR,
    TARGET_ROOT / "results/srgan_quality",
    dirs_exist_ok=True
)

print("\nUploading quality results and figures...")

try:

    kagglehub.dataset_upload(
        "niklasohlenroth/applied-ai-midterm-srgan-checkpoints",
        str(UPLOAD_ROOT),
        version_notes=(
            "Held-out SRGAN quality analysis and qualitative figures"
        )
    )

    print("\nSRGAN QUALITY BACKUP UPLOADED.")

except Exception as error:

    print("\nCloud upload failed:", repr(error))
    print("All new results remain saved locally.")

print("\n" + "=" * 55)
print("STEP 28 COMPLETE")
print("=" * 55)


In [ ]:

# ============================================================
# STEP 29 - TRAINING HISTORY FIGURES
# SRGAN and Classifier B training diagnostics
# No training required
# ============================================================

from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

PROJECT_DIR = Path("/kaggle/working/project")

BACKUP_ROOT = Path(
    "/kaggle/input/datasets/niklasohlenroth/"
    "applied-ai-midterm-srgan-checkpoints/project"
)

STEP28_ROOT = Path(
    "/kaggle/working/srgan_quality_cloud_upload/project"
)

# Use the latest complete backup.
required = [
    "results/srgan_quality/quality_summary.json",
    "results/classifier_comparison/paired_statistics.json",
    "checkpoints/model_B/model_B_finetuned_final.keras"
]

SOURCE_ROOT = None

for candidate in [STEP28_ROOT, BACKUP_ROOT]:

    if all((candidate / p).is_file() for p in required):
        SOURCE_ROOT = candidate
        break

if SOURCE_ROOT is None:
    raise FileNotFoundError(
        "Latest complete backup not found. "
        "Attach the newest private Kaggle dataset version."
    )

RESULTS_DIR = (
    PROJECT_DIR / "results/training_diagnostics"
)

VECTOR_DIR = RESULTS_DIR / "figures/vector"
RASTER_DIR = RESULTS_DIR / "figures/raster"

VECTOR_DIR.mkdir(parents=True, exist_ok=True)
RASTER_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Load training histories
# ------------------------------------------------------------

model_dir = SOURCE_ROOT / "checkpoints/model_B"

head = pd.read_csv(
    model_dir / "model_B_head_training.csv"
)

finetune = pd.read_csv(
    model_dir / "model_B_finetuning.csv"
)

srgan = pd.read_csv(
    SOURCE_ROOT / "results/srgan_training_groupnorm_v2.csv"
)

assert len(srgan) == 150
assert len(head) == 9
assert len(finetune) == 11

required_metrics = [
    "loss", "val_loss",
    "accuracy", "val_accuracy",
    "auc", "val_auc"
]

for frame in [head, finetune]:
    assert all(
        name in frame.columns
        for name in required_metrics
    )

print("=== TRAINING HISTORIES ===")
print("SRGAN epochs:", len(srgan))
print("Classifier B head epochs:", len(head))
print("Classifier B fine-tuning epochs:", len(finetune))
print("SRGAN CSV columns:", list(srgan.columns))

# ------------------------------------------------------------
# 2. Formatting
# ------------------------------------------------------------

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "figure.dpi": 120,
    "savefig.dpi": 250,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

def save_figure(fig, filename):

    fig.savefig(
        VECTOR_DIR / f"{filename}.pdf",
        bbox_inches="tight"
    )

    fig.savefig(
        VECTOR_DIR / f"{filename}.svg",
        bbox_inches="tight"
    )

    fig.savefig(
        RASTER_DIR / f"{filename}.png",
        bbox_inches="tight",
        dpi=250
    )

    plt.close(fig)

# ------------------------------------------------------------
# 3. Classifier B - Training and validation curves
# ------------------------------------------------------------

head_epochs = np.arange(1, len(head) + 1)

ft_epochs = np.arange(
    len(head) + 1,
    len(head) + len(finetune) + 1
)

best_head_epoch = int(head["val_loss"].idxmin()) + 1
best_ft_epoch = int(finetune["val_loss"].idxmin()) + 1

fig, axes = plt.subplots(
    1, 3,
    figsize=(15, 4.3)
)

metric_config = [
    ("accuracy", "Accuracy", "Accuracy"),
    ("loss", "Binary Cross-Entropy", "Loss"),
    ("auc", "ROC-AUC", "AUC")
]

for ax, (metric, title, ylabel) in zip(
    axes,
    metric_config
):

    ax.plot(
        head_epochs,
        head[metric],
        label="Training - Frozen backbone",
        linewidth=1.6
    )

    ax.plot(
        head_epochs,
        head["val_" + metric],
        label="Validation - Frozen backbone",
        linewidth=1.6,
        linestyle="--"
    )

    ax.plot(
        ft_epochs,
        finetune[metric],
        label="Training - Fine-tuning",
        linewidth=1.6
    )

    ax.plot(
        ft_epochs,
        finetune["val_" + metric],
        label="Validation - Fine-tuning",
        linewidth=1.6,
        linestyle="--"
    )

    ax.axvline(
        len(head) + 0.5,
        color="gray",
        linestyle=":",
        linewidth=1
    )

    ax.set_title(title)
    ax.set_xlabel("Total training epoch")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.2)

axes[0].legend(fontsize=7, loc="lower right")

fig.suptitle(
    "Classifier B - Frozen Backbone and Fine-tuning",
    fontsize=13
)

fig.tight_layout()

save_figure(
    fig,
    "classifier_B_training_history"
)

# ------------------------------------------------------------
# 4. SRGAN PSNR and SSIM histories
# ------------------------------------------------------------

srgan_epochs = np.arange(1, len(srgan) + 1)

def select_metric_columns(frame, keyword):

    matches = [
        c for c in frame.columns
        if keyword in c.lower()
    ]

    # Prefer validation metrics when available.
    matches.sort(
        key=lambda c: (
            "val" not in c.lower(),
            c.lower()
        )
    )

    return matches

psnr_columns = select_metric_columns(
    srgan, "psnr"
)

ssim_columns = select_metric_columns(
    srgan, "ssim"
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(11, 4.2)
)

for ax, columns, title, ylabel in zip(
    axes,
    [psnr_columns, ssim_columns],
    ["PSNR during SRGAN training",
     "SSIM during SRGAN training"],
    ["PSNR (dB)", "SSIM"]
):

    valid_count = 0

    for column in columns[:4]:

        values = pd.to_numeric(
            srgan[column],
            errors="coerce"
        )

        if values.notna().sum() == 0:
            continue

        ax.plot(
            srgan_epochs,
            values,
            label=column,
            linewidth=1.2
        )

        valid_count += 1

    if valid_count == 0:

        ax.text(
            0.5, 0.5,
            "Metric not found in training CSV",
            transform=ax.transAxes,
            ha="center",
            va="center"
        )

    else:
        ax.legend(fontsize=8)

    ax.axvline(
        107,
        linestyle="--",
        color="gray",
        linewidth=1.2,
        label="Selected epoch 107"
    )

    ax.set_xlim(1, 150)
    ax.set_xlabel("Adversarial training epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.2)

fig.suptitle(
    "SRGAN - Image Quality across 150 Epochs",
    fontsize=13
)

fig.tight_layout()

save_figure(
    fig,
    "srgan_training_quality_history"
)

# ------------------------------------------------------------
# 5. Save summary for scientific report
# ------------------------------------------------------------

summary = {
    "srgan_epochs": 150,
    "selected_srgan_epoch": 107,
    "classifier_B_head_epochs": len(head),
    "classifier_B_head_best_loss_epoch": best_head_epoch,
    "classifier_B_finetuning_epochs": len(finetune),
    "classifier_B_finetuning_best_loss_epoch": best_ft_epoch,
    "classifier_B_best_head_val_loss": float(
        head["val_loss"].min()
    ),
    "classifier_B_best_finetune_val_loss": float(
        finetune["val_loss"].min()
    ),
    "classifier_B_best_finetune_val_accuracy": float(
        finetune["val_accuracy"].max()
    ),
    "classifier_B_best_finetune_val_auc": float(
        finetune["val_auc"].max()
    ),
    "srgan_psnr_columns": psnr_columns,
    "srgan_ssim_columns": ssim_columns
}

(RESULTS_DIR / "training_summary.json").write_text(
    json.dumps(summary, indent=2)
)

print("\n=== TRAINING SUMMARY ===")
print("Classifier B head best epoch:", best_head_epoch)
print("Classifier B fine-tuning best epoch:", best_ft_epoch)
print(
    "Best fine-tuning validation accuracy:",
    summary["classifier_B_best_finetune_val_accuracy"]
)

print("\nFigures saved:")
for path in sorted(VECTOR_DIR.glob("*.pdf")):
    print(path.name)

# ------------------------------------------------------------
# 6. Cloud backup with all previous results
# ------------------------------------------------------------

UPLOAD_ROOT = Path(
    "/kaggle/working/training_diagnostics_cloud_upload"
)

if UPLOAD_ROOT.exists():
    shutil.rmtree(UPLOAD_ROOT)

TARGET_ROOT = UPLOAD_ROOT / "project"

shutil.copytree(
    SOURCE_ROOT,
    TARGET_ROOT
)

shutil.copytree(
    RESULTS_DIR,
    TARGET_ROOT / "results/training_diagnostics",
    dirs_exist_ok=True
)

assert (
    TARGET_ROOT /
    "results/srgan_quality/quality_summary.json"
).is_file()

assert (
    TARGET_ROOT /
    "results/classifier_comparison/paired_statistics.json"
).is_file()

assert (
    TARGET_ROOT /
    "checkpoints/srgan/groupnorm_v2/resume/ckpt-150.index"
).is_file()

print("\nUploading training history figures...")

try:

    kagglehub.dataset_upload(
        "niklasohlenroth/applied-ai-midterm-srgan-checkpoints",
        str(UPLOAD_ROOT),
        version_notes=(
            "Training history figures and report metadata"
        )
    )

    print("\nTRAINING HISTORY CLOUD BACKUP UPLOADED.")

except Exception as error:

    print("\nCloud upload failed:", repr(error))
    print("Figures remain available locally.")

print("\nSTEP 29 COMPLETE")
